In [15]:
pip install h3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.1 MB/s eta 0:00:00


In [16]:
# Utilities
from h3 import *
import h3
from shapely import Point, Polygon, wkt
import geopandas as gpd
import pandas as pd
import numpy as np
import networkx as nx
import geopy
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter


def h3_to_polygon(h3):
    return Polygon(h3.cell_to_boundary(h3))

def load_gdf(csv_file, index, encoding='windows-1255', crs='2039'):
    gdf = gpd.read_file(csv_file, encoding=encoding)
    gdf.set_index(index, inplace=True)

    # set geometry
    gdf['geometry'] = gdf['geometry'].apply(wkt.loads)
    gdf = gpd.GeoDataFrame(gdf, geometry='geometry', crs=crs)


In [ ]:
# read all_nodes csv file
# create gdf
#   create h3 index
#   adjust crs to 4326
#   Create polygon from h3
# read lines and planned mode
# remove old metronit nodes
# merge files to 1 gdf
# group by h3 index, node
# create ungrouped hubs csv file
# create group id
# export to groups with geometry
# read address per group
# export to grouped hubs file with address
# export to filtered hubs shp file
# add suspected nodes - assign new group.csv
# check suspected groups and alter group id
# export to filna filtered hubs csv file

# from utilities import *

# Create H3 Indices

In [17]:
# csv files encoding
encoding = 'windows-1255'

# input files
all_nodes_file = '/content/drive/MyDrive/Hubs/All_nodes+lines_29102025.csv'
lines_mode_file = '/content/drive/MyDrive/Hubs/Lines_and_Planned_Mode_13-08-2025.csv'

# set crs
crs_wgs = 4326
crs_il = 2039

# buffer for grouping, in meters
buffer = 120

gdf = gpd.read_file(all_nodes_file, encoding=encoding)
gdf.set_index('Index', inplace=True) # Corrected set_index
gdf['geometry'] = gdf['geometry'].apply(wkt.loads)
gdf = gpd.GeoDataFrame(gdf, geometry='geometry', crs=crs_il)

# Display gdf and its coordinates before reprojection
display("gdf before reprojection to WGS84:", gdf.head())
display("gdf CRS before reprojection:", gdf.crs)
display("gdf geometry coordinates before reprojection:", gdf.geometry.head().apply(lambda geom: (geom.x, geom.y)))


# Reproject to WGS84 and assign back to gdf
gdf = gdf.to_crs(crs_wgs)

# create h3 index
resolution = 10 # about 0.015 km2 or 15,000 m2, edge length = 75 meters

gdf['h3_index'] = gdf['geometry'].apply(lambda geom: h3.latlng_to_cell(geom.x, geom.y, resolution))
gdf['polygon'] = gdf['h3_index'].apply(lambda h3_index: Polygon(h3.cell_to_boundary(h3_index)))

gdf = gpd.GeoDataFrame(gdf, geometry='polygon')
gdf.set_crs(crs_wgs, inplace=True)

'gdf before reprojection to WGS84:'

,node,LINE_ID,X,Y,geometry
Index,,,,,
0,467180,4.1_Arad-BS,188658.801,578137.8485,POINT (188658.801 578137.849)
1,467180,4.1_BS-Arad,188658.801,578137.8485,POINT (188658.801 578137.849)
2,467180,4.2_LRT_BRS-Shoket,188658.801,578137.8485,POINT (188658.801 578137.849)
3,467180,4.2_LRT_Shoket-BRS,188658.801,578137.8485,POINT (188658.801 578137.849)
4,467192,4.1_Arad-BS,185010.679,575447.1948,POINT (185010.679 575447.195)


'gdf CRS before reprojection:'

<Projected CRS: EPSG:2039>
Name: Israel 1993 / Israeli TM Grid
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: Israel - onshore; Palestine Territory - onshore.
- bounds: (34.17, 29.45, 35.69, 33.28)
Coordinate Operation:
- name: Israeli TM
- method: Transverse Mercator
Datum: Israel 1993
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

'gdf geometry coordinates before reprojection:'

,geometry
Index,
0,"(188658.80097229246, 578137.8485002147)"
1,"(188658.80097229246, 578137.8485002147)"
2,"(188658.80097229246, 578137.8485002147)"
3,"(188658.80097229246, 578137.8485002147)"
4,"(185010.6790426858, 575447.1947885719)"


,node,LINE_ID,X,Y,geometry,h3_index,polygon
Index,,,,,,,
0,467180,4.1_Arad-BS,188658.801,578137.8485,POINT (34.88097 31.29451),8a3f4c15cb87fff,"POLYGON ((34.88185 31.29445, 34.88119 31.29407..."
1,467180,4.1_BS-Arad,188658.801,578137.8485,POINT (34.88097 31.29451),8a3f4c15cb87fff,"POLYGON ((34.88185 31.29445, 34.88119 31.29407..."
2,467180,4.2_LRT_BRS-Shoket,188658.801,578137.8485,POINT (34.88097 31.29451),8a3f4c15cb87fff,"POLYGON ((34.88185 31.29445, 34.88119 31.29407..."
3,467180,4.2_LRT_Shoket-BRS,188658.801,578137.8485,POINT (34.88097 31.29451),8a3f4c15cb87fff,"POLYGON ((34.88185 31.29445, 34.88119 31.29407..."
4,467192,4.1_Arad-BS,185010.679,575447.1948,POINT (34.84275 31.27015),8a3f4c02da6ffff,"POLYGON ((34.84395 31.27007, 34.84329 31.26969..."
...,...,...,...,...,...,...,...
4928,525030,BluRT2,182456.7945,644357.21,POINT (34.81335 31.89153),8a3f4dd9c0e7fff,"POLYGON ((34.81385 31.8903, 34.81319 31.88991,..."
4929,525025,BluRT2,182600.3096,643899.166,POINT (34.81488 31.8874),8a3f4dd9c18ffff,"POLYGON ((34.81554 31.88611, 34.81488 31.88572..."
4930,525020,BluRT2,182841.864,643480.8189,POINT (34.81745 31.88364),8a3f4dd9cc5ffff,"POLYGON ((34.81794 31.88319, 34.81728 31.8828,..."


In [ ]:
gdf.head(1)

,node,LINE_ID,X,Y,geometry,h3_index,polygon
Index,,,,,,,
0,467180,4.1_Arad-BS,188658.801,578137.8485,POINT (34.88097 31.29451),8a3f4c15cb87fff,"POLYGON ((34.88185 31.29445, 34.88119 31.29407..."


# change high speed rail from savidor to hashalom

In [18]:
# merge with lines_and_planned_mode file
lines = pd.read_csv(lines_mode_file, encoding=encoding)
lines.rename(columns={"Line_ModelName":"Line"}, inplace=True)
gdf = gdf.merge(lines, how='left', left_on='LINE_ID', right_on='Line')

# remove old metronit lines from file, as new metronit lines already in the file
# also, remove lrt line 15 from netanya, as there is a duplicate
def remove_lines(lines_df):
    lines_to_remove = []
    for line in lines[lines['Area']=='Haifa']['Line'].to_list():
        if line[:1] == 'm':
            lines_to_remove.append(line)

    for line in lines[lines['Area']=='Netanya']['Line'].to_list():
        if line in ['LRT151','LRT152']:
            lines_to_remove.append(line)

    return lines_to_remove

gdf = gdf[~gdf['LINE_ID'].isin(remove_lines(lines))]

In [19]:
gdf.head(1)

,node,LINE_ID,X,Y,geometry,h3_index,polygon,Line,Mode_Planned,Line_Name,Line_Description,Area
0,467180,4.1_Arad-BS,188658.801,578137.8485,POINT (34.88097 31.29451),8a3f4c15cb87fff,"POLYGON ((34.88185 31.29445, 34.88119 31.29407...",4.1_Arad-BS,LRT,ערד-באר שבע מערב,ערד-באר שבע מערב,Beer Sheva


# Create Groups

In [20]:
# 2 step grouping phase
# 1. group by h3_index and mode planned
# 2. group the first step only by h3_index

hubs = gdf.groupby(['h3_index','node','Mode_Planned','Area']).agg({'Line': ['nunique','unique']})
hubs.reset_index(inplace=True)
# export to csv - why?
# hubs.to_csv('hubs.csv', encoding=encoding)

# create h3 index polygons
hubs['geometry'] = hubs['h3_index'].apply(lambda x: h3.cell_to_boundary(x))
hubs['geometry'] = hubs['geometry'].apply(Polygon)
gdf = gpd.GeoDataFrame(hubs, geometry='geometry', crs=crs_wgs)

# flat indices
cols = ['h3_index','node','Mode_Planned','Area','Line_Nunique','Line_Unique','geometry']
gdf.columns = gdf.columns.to_flat_index()
gdf.columns = cols

# create group id
# reproject to a projected crs for israel (units in meters)
gdf = gdf.to_crs(crs_il)
# create a buffer polygon
gdf['buffer'] = gdf.geometry.buffer(buffer)

In [21]:
gdf.head(1)

,h3_index,node,Mode_Planned,Area,Line_Nunique,Line_Unique,geometry,buffer
0,8a2da4100a27fff,31655,Interurban Rail,National,2,"[rail_1_1, rail_1_2]","POLYGON ((254717.95 790579.037, 254657.43 7905...","POLYGON ((254837.895 790575.414, 254836.911 79..."


In [22]:
gdf.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 1453 entries, 0 to 1452
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   h3_index      1453 non-null   object  
 1   node          1453 non-null   object  
 2   Mode_Planned  1453 non-null   object  
 3   Area          1453 non-null   object  
 4   Line_Nunique  1453 non-null   int64   
 5   Line_Unique   1453 non-null   object  
 6   geometry      1453 non-null   geometry
 7   buffer        1453 non-null   geometry
dtypes: geometry(2), int64(1), object(5)
memory usage: 90.9+ KB


In [23]:
gdf['Line_Nunique'] = gdf['Line_Nunique'].astype('int64')

In [24]:
def buffer_grouping(gdf):

    # set spatial index
    sindex = gdf.sindex

    # Initialize Graph
    G = nx.Graph()

    for idx, geom in gdf['buffer'].items():
        G.add_node(idx)
        possible_matches_index = list(sindex.intersection(geom.bounds))
        # check if the same lines are on both nodes
        for pm in possible_matches_index:
            lines_pm = gdf.iloc[pm, 5]
            lines_idx = gdf.iloc[idx, 5]

            if pm != idx and set(lines_pm) == set(lines_idx):
                possible_matches_index.remove(pm)

        for match_idx in possible_matches_index:
            lines_match_idx = gdf.iloc[match_idx, 5]
            lines_idx = gdf.iloc[idx, 5]

            if match_idx != idx and set(lines_match_idx) == set(lines_idx):
                possible_matches_index.remove(match_idx)
            if geom.intersects(gdf['buffer'].loc[match_idx]):
                G.add_edge(idx, match_idx)

    # Assign group ids based on connected components
    gdf['group'] = -1
    for group_id, component in enumerate(nx.connected_components(G)):
        gdf.loc[list(component), 'group'] = group_id

    # Drop the buffer column after group ids been assigned
    gdf.drop(columns='buffer', inplace=True)

    return gdf


In [25]:
# apply grouping
gdf = buffer_grouping(gdf)

In [26]:
gdf.head(1)

,h3_index,node,Mode_Planned,Area,Line_Nunique,Line_Unique,geometry,group
0,8a2da4100a27fff,31655,Interurban Rail,National,2,"[rail_1_1, rail_1_2]","POLYGON ((254717.95 790579.037, 254657.43 7905...",0


# Geocoding - address

In [27]:
# apply geocoding to add address for each h3_index
# first step - create centroid and x,y for each h3 index
# last step - run reverse geocode

# make sure gdf is in the right crs for geocoding
gdf = gdf.to_crs(epsg=crs_wgs)

# Initialize geocoder
geolocator = Nominatim(user_agent="h3_geocoder")
geocode = RateLimiter(geolocator.reverse, min_delay_seconds=1) # Corrected typo

# Get centroid for each polygon for geocoding
gdf['centroid'] = gdf.geometry.centroid
gdf['lat'] = gdf.centroid.y
gdf['lon'] = gdf.centroid.x

# Reverse geocode function
def reverse_geocode(row):
    location = geocode((row['lat'], row['lon']), exactly_one=True)
    return location.address if location else None

# apply reverse geocoding on appropriate column
gdf['address'] = gdf.apply(reverse_geocode, axis=1)

# drop temporary columns
gdf.drop(columns=['centroid','lat','lon'], inplace=True)

# review nodes, some groups are suspicious, too close or including 2 stops of the same line in the group
# should be applied, but not now - need to review the code in whole after run
# there's an old version of a file with nodes and groups that are suspicious and needed to get a new group id
# new_group = pd.read_csv('suspected_nodes_assign_new_group.csv')
# gdf['node'] = gdf['node'].astype('int64') # make sure the node column is of type int
# for idx, row in new_group.iterrows():
#     gdf.loc[gdf['node']==row['node'], 'group'] = row['NewGroup']



# at this point, getting the groups file and adding data:
# LandOwnership
# Region (haifa, tlv etc.)
# Location (galin, inner, outer)
# Boardings
# Alightings
# TotalDemand

####################
## Hubs Filtering ##
####################

/tmp/ipython-input-2605157229.py:13: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf['centroid'] = gdf.geometry.centroid
/tmp/ipython-input-2605157229.py:14: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf['lat'] = gdf.centroid.y
/tmp/ipython-input-2605157229.py:15: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf['lon'] = gdf.centroid.x


In [28]:
# save the hubs list with group ids
processed_rows = []

for index, row in gdf.iterrows():
    processed_row = row.copy()
    for col, value in row.items():
        if isinstance(value, str):
            try:
                # Attempt to encode with windows-1255
                value.encode('windows-1255')
            except UnicodeEncodeError:
                try:
                    # If windows-1255 fails, attempt to encode with utf-8
                    processed_row[col] = value.encode('utf-8', errors='replace').decode('utf-8')
                except Exception as e:
                    # Handle cases where even utf-8 might have issues (less likely, but for robustness)
                    print(f"Could not encode value in row {index}, column {col}: {value} with utf-8. Error: {e}")
                    # Optionally replace with a placeholder or handle as needed
                    processed_row[col] = "Encoding Error"
    processed_rows.append(processed_row)

gdf = pd.DataFrame(processed_rows)


In [29]:
gdf.to_csv('/content/drive/MyDrive/Hubs/groups_hubs_29102025.csv', encoding='utf-8', index=False)

# Finish setting base set
# Reload csv

In [30]:
gdf = gpd.read_file('/content/drive/MyDrive/Hubs/groups_hubs_29102025.csv', encoding='utf-8')

In [31]:
from shapely import wkt
import geopandas as gpd

# Convert the string representations of geometries to shapely objects
gdf['geometry'] = gdf['geometry'].apply(wkt.loads)

# Create the GeoDataFrame with the correct geometry objects
gdf = gpd.GeoDataFrame(gdf, geometry='geometry', crs=crs_il)

# Add Data: Location
first, use the layer: metro_2008 to tag nodes within metro areas and the exact ring
the nodes left without data, will be tagged using the layer: districts
just tag with district.


In [32]:
gdf.head(3)

,h3_index,node,Mode_Planned,Area,Line_Nunique,Line_Unique,geometry,group,address
0,8a2da4100a27fff,31655,Interurban Rail,National,2,['rail_1_1' 'rail_1_2'],"POLYGON ((35.583 33.21, 35.582 33.21, 35.581 3...",0,"מנחם בגין/נורית, שדרות מנחם בגין, הורדים, קרית..."
1,8a2da4c22257fff,36963,BRT,Haifa,1,['brt022'],"POLYGON ((35.327 32.926, 35.326 32.925, 35.325...",1,"מיי סנטר, מעלה כמון, אזור תעשייה כרמיאל, אזור ..."
2,8a2da4c22627fff,36959,BRT,Haifa,1,['brt021'],"POLYGON ((35.317 32.922, 35.316 32.922, 35.316...",2,"החרושת/חשמל, החרושת, אזור תעשייה כרמיאל, מרכז ..."


In [33]:
# rename Area column to Model column, not to mistake with the Area tagging for location
gdf.rename(columns={'Area': 'Model'}, inplace=True)

In [34]:
# Load layers
metro_shp = '/content/drive/MyDrive/Hubs/Location/metro_2008.shp'
districts_shp = '/content/drive/MyDrive/Hubs/Location/Districts.shp'

In [35]:
def load_metro(shp):
    metro = gpd.read_file(shp, encoding='windows-1255')
    metro_names = {'חיפ': 'חיפה',
                   'ירושלי': 'ירושלים',
                   'תל אבי': 'תל אביב',
                   'באר שב': 'באר שבע'}

    metro['METRO_NAME'] = metro['METRO_NAME'].replace(metro_names)
    metro = metro[['METRO_CODE','ZONE_CODE','METRO_NAME','ZONE_NAME','geometry']]

    return metro

In [36]:
metro = load_metro('/content/drive/MyDrive/Hubs/Location/metro_2008.shp')

In [37]:
def load_districts(shp):
    districts = gpd.read_file(shp, encoding='utf-8')
    districts = districts[['ID','MACHOZ','geometry']]

    # remove 'מחוז' from the string of the MACHOZ column
    districts['MACHOZ'] = districts['MACHOZ'].str.replace('מחוז', '').str.strip()

    return districts

In [38]:
districts = load_districts('/content/drive/MyDrive/Hubs/Location/Districts.shp')

In [39]:
districts.head(3)

,ID,MACHOZ,geometry
0,1,דרום,"POLYGON ((34.67496 31.84917, 34.67504 31.84915..."
1,2,חיפה,"MULTIPOLYGON (((35.15462 32.82356, 35.15452 32..."
2,3,ירושלים,"POLYGON ((35.22034 31.88116, 35.2204 31.88109,..."


In [40]:
def tag_with_spatial_data(gdf, metro_shp_path, districts_shp_path, crs_wgs=4326):
    """
    Tags a GeoDataFrame with location information from metro and districts shapefiles.

    Args:
        gdf (gpd.GeoDataFrame): The input GeoDataFrame to tag.
        metro_shp_path (str): Path to the metro shapefile.
        districts_shp_path (str): Path to the districts shapefile.
        crs_wgs (int): The WGS 84 CRS code (default is 4326).

    Returns:
        gpd.GeoDataFrame: The GeoDataFrame with 'area' and 'location' columns added and cleaned.
    """
    # Ensure gdf is a GeoDataFrame and set CRS
    # Explicitly set the active geometry column
    gdf = gpd.GeoDataFrame(gdf, geometry='geometry', crs=crs_wgs)
    gdf.set_geometry('geometry', inplace=True)


    # Load spatial data
    metro = load_metro(metro_shp_path)
    districts_gdf = load_districts(districts_shp_path) # Use a different variable name

    # Ensure metro and districts are in the correct CRS for spatial joins
    metro = metro.to_crs(epsg=crs_wgs)
    # Reproject districts using the new variable name
    districts_gdf = districts_gdf.to_crs(epsg=crs_wgs)


    # Drop 'index_right' column if it exists (from previous joins)
    if 'index_right' in gdf.columns:
        gdf.drop(columns=['index_right'], inplace=True)

    # Drop 'area' and 'location' columns if they exist (from previous runs)
    if 'area' in gdf.columns:
        gdf.drop(columns=['area'], inplace=True)
    if 'location' in gdf.columns:
        gdf.drop(columns=['location'], inplace=True)

    # Create a new GeoDataFrame with essential columns and preserve the original index as a column for the first join
    gdf_for_join = gdf[['geometry']].copy()
    gdf_for_join = gdf_for_join.reset_index() # Reset index without dropping

    # Perform a spatial join with metro to tag 'area' and 'location'
    gdf_joined_metro = gpd.sjoin(gdf_for_join, metro[['METRO_NAME','ZONE_NAME', 'geometry']], how="left", predicate="intersects")

    # Rename the joined column to 'area' and 'location'
    gdf_joined_metro.rename(columns={'METRO_NAME': 'area', 'ZONE_NAME': 'location'}, inplace=True)

    # Combine gdf and gdf_joined_metro based on the original index
    # Use pd.concat after setting 'Index' as the index in gdf_joined_metro
    gdf_joined_metro = gdf_joined_metro.set_index('index') # Corrected column name to 'Index'
    gdf = pd.concat([gdf, gdf_joined_metro[['area', 'location']]], axis=1)


    # Tag rows that are still NaN in 'area' with district name
    # Select indices of rows in gdf where 'area' is NaN
    nan_area_indices = gdf[gdf['area'].isna()].index

    # Select rows in gdf where 'area' is NaN for the second join
    gdf_nan_area = gdf.loc[nan_area_indices].copy()

    # Ensure gdf_nan_area is a GeoDataFrame for the second join
    gdf_nan_area = gpd.GeoDataFrame(gdf_nan_area, geometry='geometry', crs=crs_wgs)
    gdf_nan_area.set_geometry('geometry', inplace=True) # Explicitly set active geometry


    # Perform a spatial join with districts for these rows, using the new variable name
    gdf_tagged_districts = gpd.sjoin(gdf_nan_area, districts_gdf[['MACHOZ', 'geometry']], how="left", predicate="within", rsuffix='_district')


    # Update the original gdf with the MACHOZ values using .loc
    # Iterate through the tagged districts and update the original gdf by index
    for index, row in gdf_tagged_districts.iterrows():
        if pd.notna(row['MACHOZ']):
            gdf.loc[index, 'area'] = row['MACHOZ']
            gdf.loc[index, 'location'] = row['MACHOZ'] # Assuming location should also be updated with district for these rows

    # Identify the columns to keep for the cleaned DataFrame
    columns_to_keep = ['h3_index', 'node', 'Mode_Planned', 'Model', 'Line_Nunique', 'Line_Unique', 'geometry', 'group', 'address', 'area', 'location'] # Corrected column names

    # Return a new DataFrame with only the desired columns
    # Ensure all columns_to_keep exist in gdf before selecting
    cols_exist = [col for col in columns_to_keep if col in gdf.columns]
    gdf_cleaned = gdf[cols_exist].copy()


    return gdf_cleaned

In [41]:
gdf.head()

,h3_index,node,Mode_Planned,Model,Line_Nunique,Line_Unique,geometry,group,address
0,8a2da4100a27fff,31655,Interurban Rail,National,2,['rail_1_1' 'rail_1_2'],"POLYGON ((35.583 33.21, 35.582 33.21, 35.581 3...",0,"מנחם בגין/נורית, שדרות מנחם בגין, הורדים, קרית..."
1,8a2da4c22257fff,36963,BRT,Haifa,1,['brt022'],"POLYGON ((35.327 32.926, 35.326 32.925, 35.325...",1,"מיי סנטר, מעלה כמון, אזור תעשייה כרמיאל, אזור ..."
2,8a2da4c22627fff,36959,BRT,Haifa,1,['brt021'],"POLYGON ((35.317 32.922, 35.316 32.922, 35.316...",2,"החרושת/חשמל, החרושת, אזור תעשייה כרמיאל, מרכז ..."
3,8a2da4c2275ffff,36961,BRT,Haifa,2,['brt021' 'brt022'],"POLYGON ((35.319 32.924, 35.318 32.923, 35.318...",2,"החרושת, אזור תעשייה כרמיאל, אזור תעשׂייה, כרמיא..."
4,8a2da4c24987fff,36924,BRT,Haifa,4,['brt011' 'brt012' 'brt021' 'brt022'],"POLYGON ((35.347 32.862, 35.346 32.862, 35.346...",3,"طريق الجليل, عرابة, נפת עכו, מחוז הצפון, 20176..."


In [42]:
gdf = tag_with_spatial_data(gdf, metro_shp, districts_shp, crs_wgs=4326)

In [43]:
gdf.head()

,h3_index,node,Mode_Planned,Model,Line_Nunique,Line_Unique,geometry,group,address,area,location
0,8a2da4100a27fff,31655,Interurban Rail,National,2,['rail_1_1' 'rail_1_2'],"POLYGON ((35.58265 33.21008, 35.582 33.20969, ...",0,"מנחם בגין/נורית, שדרות מנחם בגין, הורדים, קרית...",צפון,צפון
1,8a2da4c22257fff,36963,BRT,Haifa,1,['brt022'],"POLYGON ((35.32671 32.92555, 35.32606 32.92516...",1,"מיי סנטר, מעלה כמון, אזור תעשייה כרמיאל, אזור ...",צפון,צפון
2,8a2da4c22627fff,36959,BRT,Haifa,1,['brt021'],"POLYGON ((35.31703 32.92236, 35.31638 32.92197...",2,"החרושת/חשמל, החרושת, אזור תעשייה כרמיאל, מרכז ...",צפון,צפון
3,8a2da4c2275ffff,36961,BRT,Haifa,2,['brt021' 'brt022'],"POLYGON ((35.31899 32.92353, 35.31834 32.92314...",2,"החרושת, אזור תעשייה כרמיאל, אזור תעשׂייה, כרמיא...",צפון,צפון
4,8a2da4c24987fff,36924,BRT,Haifa,4,['brt011' 'brt012' 'brt021' 'brt022'],"POLYGON ((35.34685 32.86202, 35.3462 32.86163,...",3,"طريق الجليل, عرابة, נפת עכו, מחוז הצפון, 20176...",צפון,צפון


In [44]:
# handle nan
# Correct spelling of location

In [45]:
locations = {l: l for l in gdf['location'].unique()}
locations['טבעת חיצוני'] = 'טבעת חיצונית'
locations['טבעת פנימי'] = 'טבעת פנימית'
locations['גלעי'] = 'גלעין'
locations['טבעת תיכונ'] = 'טבעת תיכונה'

In [46]:
def rename_location(location):
    return locations.get(location, location)

gdf['location'] = gdf['location'].apply(rename_location)

In [47]:
gdf[gdf['location'].isna()]
# no significant, can be related to 'Merkaz'

,h3_index,node,Mode_Planned,Model,Line_Nunique,Line_Unique,geometry,group,address,area,location
1264,8a3f4dd76507fff,195251,BRT,Tel Aviv,1,['BRT-1'],"POLYGON ((34.99558 32.10529, 34.99492 32.1049,...",901,"חוצה שומרון, ראש העין, נפת פתח תקווה, מחוז המר...",NaN,NaN


In [48]:
gdf.to_csv('/content/drive/MyDrive/Hubs/groups_hubs_with_location_29102025.csv', encoding='utf-8', index=False)

# Add Daily Demand

In [49]:
# merge all_nodes demand data into 1 dataframe
# join using node id and area

In [50]:
import folium

In [51]:
# Create a base map
# You might want to center the map based on the data's extent
# For now, let's use a general center point (e.g., approximate center of Israel)
map_center = [31.77, 35.22] # Approximate center of Israel
m = folium.Map(location=map_center, zoom_start=8)

In [52]:
# Add the gdf as a GeoJson layer to the map
# Ensure gdf is in WGS84 (EPSG:4326) for folium
# Use gdf_cleaned which has the correct columns after processing
gdf_wgs84 = gdf.to_crs(epsg=4326)

# Calculate centroids
gdf_wgs84['centroid'] = gdf_wgs84.geometry.centroid

# Create a FeatureGroup to add markers to
marker_group = folium.FeatureGroup(name='Hub Centroids')

# Add markers for each centroid
for index, row in gdf_wgs84.iterrows():
    if row['centroid'] is not None: # Check if centroid calculation was successful
        # Create a Tooltip
        tooltip_text = f"Group: {row['group']}"
        tooltip = folium.Tooltip(tooltip_text)

        folium.Marker(
            location=[row['centroid'].y, row['centroid'].x],
            tooltip=tooltip # Add the tooltip
        ).add_to(marker_group)

# Add the marker group to the map
marker_group.add_to(m)

# Add layer control to toggle centroid markers
folium.LayerControl().add_to(m)

/tmp/ipython-input-2951369840.py:7: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_wgs84['centroid'] = gdf_wgs84.geometry.centroid


In [53]:

# Display the map
m

In [54]:
xl = pd.ExcelFile('/content/drive/MyDrive/Hubs/Nodes_w_results_21082025.xlsx')

In [55]:
xl.sheet_names

['Params',
 '15040',
 '25040',
 '35040',
 '45040',
 '5040_Daily',
 '15087',
 '25087',
 '35087',
 'Daily_5087',
 'Daily_BS',
 'Daily_Hadera',
 'HaifaNewMetronit',
 'Daily_Jerusalem',
 'National',
 '15093',
 '25093',
 '35093',
 'Daily_5093']

In [56]:
sheet_names = {'5040_Daily': 'Haifa',
 'Daily_5087': 'Tel Aviv',
 'Daily_BS': 'Beer Sheva',
 'Daily_Hadera': 'Hadera',
 'Daily_Jerusalem': 'Jerusalem',
 'HaifaNewMetronit': 'Haifa Metronit',
 'Daily_5093': 'Ashdod-Ashkelon',
               'National': 'Rail'}

In [57]:
demand = {sheet: pd.read_excel(xl, sheet) for sheet in xl.sheet_names}

In [58]:
# Iterate over a copy of the keys to avoid the RuntimeError
for key in list(demand.keys()):
  if key in sheet_names.keys():
    demand[sheet_names[key]] = demand.pop(key)

In [59]:
for key in list(demand.keys()):
  if key not in sheet_names.values():
    demand.pop(key)

In [60]:
demand.keys()

dict_keys(['Haifa', 'Tel Aviv', 'Beer Sheva', 'Hadera', 'Haifa Metronit', 'Jerusalem', 'Rail', 'Ashdod-Ashkelon'])

In [61]:
demand['Beer Sheva'].rename(columns={'NODE_ID': 'Node'}, inplace=True)
demand['Beer Sheva']['TotalTransfers'] = 0

In [62]:
demand['Beer Sheva'].head(1)

,Node,Boardings_Daily,Alightings_Daily,TotalTransfers
0,463642,6401.0,13275.0,0


In [63]:
demand['Hadera'].head(1)

,hub_name,NodeID,On,Off
0,Hadera_east,984885.0,16945.135129,17000


In [64]:
demand['Hadera'].rename(columns={'NodeID': 'Node',
                                 'On': 'Boardings_Daily',
                                 'Off': 'Alightings_Daily'},
                        inplace=True)
demand['Hadera']['TotalTransfers'] = 0
drop_cols = ['hub_name']
demand['Hadera'].drop(columns=drop_cols, inplace=True)

In [65]:
demand['Hadera'].head(1)

,Node,Boardings_Daily,Alightings_Daily,TotalTransfers
0,984885.0,16945.135129,17000,0


In [66]:
demand['Tel Aviv'].head(1)

,Node,NumLines,InitialBoardings,TransferBoardings,TotalBoardings,FinalAlight,TransferAlight,TotalAlight,TotalTransfers,Unnamed: 9
0,100,NaN,0.0,0.0,0.0,6.1,4.3,0.0,4.3,NaN


In [67]:
demand['Tel Aviv'].rename(columns={'TotalBoardings': 'Boardings_Daily',
                                   'TotalAlight': 'Alightings_Daily'},
                                   inplace=True)
drop_cols = ['NumLines','InitialBoardings','TransferBoardings','FinalAlight','TransferAlight','Unnamed: 9']
demand['Tel Aviv'].drop(columns=drop_cols, inplace=True)

In [68]:
demand['Tel Aviv'].head(1)

,Node,Boardings_Daily,Alightings_Daily,TotalTransfers
0,100,0.0,0.0,4.3


In [69]:
demand['Ashdod-Ashkelon'].head(1)

,Node,InitialBoardings,FinalAlight
0,100,0.0,9.770383


In [70]:
demand['Ashdod-Ashkelon'].rename(columns={'InitialBoardings': 'Boardings_Daily',
                                      'FinalAlight': 'Alightings_Daily'},
                             inplace=True)
demand['Ashdod-Ashkelon']['TotalTransfers'] = 0
demand['Ashdod-Ashkelon'].head(1)

,Node,Boardings_Daily,Alightings_Daily,TotalTransfers
0,100,0.0,9.770383,0


In [71]:
demand['Haifa'].head(1)

,Node,InitialBoardings,TransferBoardings,TotalBoardings,FinalAlight,TransferAlight,TotalAlight
0,51,0.0,0.0,0.0,0.0,0.0,0.0


In [72]:
demand['Haifa'].rename(columns={'TotalBoardings': 'Boardings_Daily',
                                'TotalAlight': 'Alightings_Daily'},
                       inplace=True)
demand['Haifa']['TotalTransfers'] = demand['Haifa']['TransferBoardings'] + demand['Haifa']['TransferAlight']
drop_cols = ['InitialBoardings','TransferBoardings','FinalAlight','TransferAlight']
demand['Haifa'].drop(columns=drop_cols, inplace=True)

In [73]:
demand['Haifa'].head(1)

,Node,Boardings_Daily,Alightings_Daily,TotalTransfers
0,51,0.0,0.0,0.0


In [74]:
demand['Haifa Metronit'].head(1)

,NewNode,ModelNode,Boardings_Daily,Alightings_Daily,TotalDemand_Daily
0,1000,NoData,NaN,NaN,NaN


In [75]:
demand['Haifa Metronit'].rename(columns={'ModelNode': 'Node'}, inplace=True)
demand['Haifa Metronit']['TotalTransfers'] = 0
drop_cols = ['NewNode','TotalDemand_Daily']
demand['Haifa Metronit'].drop(columns=drop_cols, inplace=True)

In [76]:
demand['Haifa Metronit'].head(1)

,Node,Boardings_Daily,Alightings_Daily,TotalTransfers
0,NoData,NaN,NaN,0


In [77]:
demand['Jerusalem'].head(1)

,ID,F_station,name,SUM_SUM_F_aligh,SUM_SUM_BOARD,DailyAlight,DailyDemand,DailyBoard,DailyAlight.1,DailyBoard.1,AlightORBoardings_ByDemand,Balanced_Daily_Alight,Unnamed: 12,DailyBoard_2050,DailyAlight_2050,Unnamed: 15,1.025
0,51001,Heil-HaAvir,חיל האוויר,146.675323,494.748379,1760.103876,7697.084424,3848.542212,3848.542212,5936.980548,5936.980548,4488.068926,NaN,4926.459403,4926.459403,NaN,NaN


In [78]:
demand['Jerusalem'].rename(columns={'ID': 'Node',
                                    'DailyBoard_2050': 'Boardings_Daily',
                                    'DailyAlight_2050': 'Alightings_Daily'},
                           inplace=True)
demand['Jerusalem']['TotalTransfers'] = 0
keep_cols = ['Node','Boardings_Daily','Alightings_Daily','TotalTransfers']
demand['Jerusalem'] = demand['Jerusalem'][keep_cols]

In [79]:
demand['Jerusalem'].head(1)

,Node,Boardings_Daily,Alightings_Daily,TotalTransfers
0,51001,4926.459403,4926.459403,0


In [80]:
# add model source column
for key in list(demand.keys()):
  demand[key]['Model'] = key

In [81]:
# before concatinating all the demand worksheets,
# need to add to each demand a column with its name ("hadera", "haifa" etc)
# need to rename boarding and alighting columns
# need to decide on a transfer columns for when there is one

In [82]:
combined = pd.concat(demand.values(), ignore_index=True, axis=0)

In [83]:
combined

,Node,Boardings_Daily,Alightings_Daily,TotalTransfers,Model
0,51,0.000000,0.000000,0.0,Haifa
1,52,0.000000,0.000000,0.0,Haifa
2,53,0.000000,0.000000,0.0,Haifa
3,54,0.000000,0.000000,0.0,Haifa
4,55,0.000000,0.000000,0.0,Haifa
...,...,...,...,...,...
40488,523021,2104.113337,1882.167550,0.0,Ashdod-Ashkelon
40489,523022,19.817225,29.862709,0.0,Ashdod-Ashkelon
40490,523023,4497.052560,4505.246070,0.0,Ashdod-Ashkelon
40491,523024,1341.540125,1209.761491,0.0,Ashdod-Ashkelon


In [84]:
combined.to_csv('/content/drive/MyDrive/Hubs/demand_by_node_29102025.csv', encoding='utf-8', index=False)

In [85]:
gdf['Model'].unique()

array(['National', 'Haifa', 'Beer Sheva', 'Ashdod-ashkelon', 'Tel Aviv',
       'Jerusalem', 'Netanya'], dtype=object)

In [86]:
combined['Model'].unique()

array(['Haifa', 'Tel Aviv', 'Beer Sheva', 'Hadera', 'Haifa Metronit',
       'Jerusalem', 'Ashdod-Ashkelon'], dtype=object)

In [ ]:
# first - assign demand from the models: Haifa, Tel Aviv, Beer Sheva, Jerusalem, Ashdod-Ashkelon
# then, update demand for the Nodes in the models: Haifa Metronit, Hadera
# last, check the Netanya data, check national model data for specific locations, update Rail

In [87]:
gdf.head(1)

,h3_index,node,Mode_Planned,Model,Line_Nunique,Line_Unique,geometry,group,address,area,location
0,8a2da4100a27fff,31655,Interurban Rail,National,2,['rail_1_1' 'rail_1_2'],"POLYGON ((35.58265 33.21008, 35.582 33.20969, ...",0,"מנחם בגין/נורית, שדרות מנחם בגין, הורדים, קרית...",צפון,צפון


In [92]:
# create total demand column and transfers column
gdf['TotalDemand'] = 0
gdf['TotalTransfers'] = 0

In [93]:
gdf.to_csv('/content/drive/MyDrive/Hubs/groups_hubs_with_demand_29102025.csv', encoding='utf-8', index=False)

In [94]:
gdf = gpd.read_file('/content/drive/MyDrive/Hubs/groups_hubs_with_demand_29102025.csv', encoding='utf-8')

# Explicitly convert TotalDemand and TotalTransfers to numeric after loading
gdf['TotalDemand'] = pd.to_numeric(gdf['TotalDemand'], errors='coerce').fillna(0)
gdf['TotalTransfers'] = pd.to_numeric(gdf['TotalTransfers'], errors='coerce').fillna(0)
gdf['Line_Nunique'] = pd.to_numeric(gdf['Line_Nunique'], errors='coerce').fillna(0)

In [95]:
gdf.head(5)

,h3_index,node,Mode_Planned,Model,Line_Nunique,Line_Unique,geometry,group,address,area,location,TotalDemand,TotalTransfers
0,8a2da4100a27fff,31655,Interurban Rail,National,2,['rail_1_1' 'rail_1_2'],"POLYGON ((35.58265301880704 33.2100839260914, ...",0,"מנחם בגין/נורית, שדרות מנחם בגין, הורדים, קרית...",צפון,צפון,0,0
1,8a2da4c22257fff,36963,BRT,Haifa,1,['brt022'],"POLYGON ((35.32671223620188 32.92554871232436,...",1,"מיי סנטר, מעלה כמון, אזור תעשייה כרמיאל, אזור ...",צפון,צפון,0,0
2,8a2da4c22627fff,36959,BRT,Haifa,1,['brt021'],POLYGON ((35.31703160566268 32.922357121183495...,2,"החרושת/חשמל, החרושת, אזור תעשייה כרמיאל, מרכז ...",צפון,צפון,0,0
3,8a2da4c2275ffff,36961,BRT,Haifa,2,['brt021' 'brt022'],"POLYGON ((35.31899096754 32.923530411401764, 3...",2,"החרושת, אזור תעשייה כרמיאל, אזור תעשׂייה, כרמיא...",צפון,צפון,0,0
4,8a2da4c24987fff,36924,BRT,Haifa,4,['brt011' 'brt012' 'brt021' 'brt022'],"POLYGON ((35.34684915390923 32.86202464664849,...",3,"طريق الجليل, عرابة, נפת עכו, מחוז הצפון, 20176...",צפון,צפון,0,0


In [96]:
gdf[gdf['node']=='511248']

,h3_index,node,Mode_Planned,Model,Line_Nunique,Line_Unique,geometry,group,address,area,location,TotalDemand,TotalTransfers
1128,8a3f4dc7052ffff,511248,LRT,Netanya,4,['LRT121' 'LRT122' 'LRT131' 'LRT132'],"POLYGON ((34.84088769100656 32.21510746276794,...",774,"איילון צפון, מועצה אזורית חוף השרון, נפת השרון...",תל אביב,טבעת חיצונית,0,0


In [ ]:
# update Haifa Metronit & Hadera

In [97]:
demand['Hadera']

,Node,Boardings_Daily,Alightings_Daily,TotalTransfers,Model
0,984885.0,16945.135129,17000,0,Hadera
1,984883.0,79459.693944,80000,0,Hadera
2,984890.0,79459.693944,80000,0,Hadera
3,NaN,20046.359359,20000,0,Hadera
4,984889.0,46514.103109,46500,0,Hadera
5,987294.0,14459.873266,14500,0,Hadera


In [98]:
gdf[gdf['node']==987294]

,h3_index,node,Mode_Planned,Model,Line_Nunique,Line_Unique,geometry,group,address,area,location,TotalDemand,TotalTransfers


In [99]:
# update Hadera using Hadera demand data

# Ensure 'Node' column in demand['Hadera'] is numeric
demand['Hadera']['Node'] = pd.to_numeric(demand['Hadera']['Node'], errors='coerce')

# Select relevant columns from demand['Hadera']
hadera_demand = demand['Hadera'][['Node', 'Boardings_Daily', 'Alightings_Daily', 'TotalTransfers']]

# Aggregate hadera_demand by Node, summing the demand and transfers
hadera_demand_agg = hadera_demand.groupby('Node').agg(
    Boardings_Hadera_sum=('Boardings_Daily', 'sum'),
    Alightings_Hadera_sum=('Alightings_Daily', 'sum'),
    TotalTransfers_Hadera_sum=('TotalTransfers', 'sum')
).reset_index() # Reset index to make 'Node' a column again

# Rename 'Node' to 'node' for matching with gdf
hadera_demand_agg.rename(columns={'Node': 'node'}, inplace=True)

# Ensure 'node' column in gdf is numeric for matching
gdf['node'] = pd.to_numeric(gdf['node'], errors='coerce')

# Set 'node' as index in hadera_demand_agg for efficient lookups
hadera_demand_agg.set_index('node', inplace=True)

# Display data types before the problematic line
display("gdf dtypes before update:", gdf[['TotalDemand', 'TotalTransfers']].dtypes)
display("hadera_demand_agg dtypes before update:", hadera_demand_agg[['Boardings_Hadera_sum', 'Alightings_Hadera_sum', 'TotalTransfers_Hadera_sum']].dtypes)


# Update TotalDemand and TotalTransfers in gdf using data from hadera_demand_agg
# Iterate through gdf and update rows where node matches in hadera_demand_agg
for index, row in gdf.iterrows():
    if row['node'] in hadera_demand_agg.index:
        # Get corresponding demand data from hadera_demand_agg
        hadera_data = hadera_demand_agg.loc[row['node']]

        # Update TotalDemand and TotalTransfers, adding to existing values
        gdf.loc[index, 'TotalDemand'] = gdf.loc[index, 'TotalDemand'] + hadera_data['Boardings_Hadera_sum'] + hadera_data['Alightings_Hadera_sum']
        gdf.loc[index, 'TotalTransfers'] = gdf.loc[index, 'TotalTransfers'] + hadera_data['TotalTransfers_Hadera_sum']


# Drop the aggregated columns from Hadera demand if they were added in a previous step
# Use a try-except block to handle cases where these columns might not exist
try:
    gdf.drop(columns=['Boardings_Hadera_sum', 'Alightings_Hadera_sum', 'TotalTransfers_Hadera_sum'], inplace=True)
except KeyError:
    pass # Columns not found, no need to drop


# Display the head of gdf to check the updated columns (optional)
# display(gdf.head())

'gdf dtypes before update:'

,0
TotalDemand,int64
TotalTransfers,int64


'hadera_demand_agg dtypes before update:'

,0
Boardings_Hadera_sum,float64
Alightings_Hadera_sum,int64
TotalTransfers_Hadera_sum,int64


/tmp/ipython-input-3268646486.py:38: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '159459.6939435' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  gdf.loc[index, 'TotalDemand'] = gdf.loc[index, 'TotalDemand'] + hadera_data['Boardings_Hadera_sum'] + hadera_data['Alightings_Hadera_sum']


In [100]:
gdf.head(1)

,h3_index,node,Mode_Planned,Model,Line_Nunique,Line_Unique,geometry,group,address,area,location,TotalDemand,TotalTransfers
0,8a2da4100a27fff,31655,Interurban Rail,National,2,['rail_1_1' 'rail_1_2'],"POLYGON ((35.58265301880704 33.2100839260914, ...",0,"מנחם בגין/נורית, שדרות מנחם בגין, הורדים, קרית...",צפון,צפון,0.0,0


# update this part, as 'Rail' is no longer a stand alone name, should use the 3 different kinds of 'Rail' or - read the end of the string where 'Rail' is

In [101]:
gdf['area'].unique()

array(['צפון', 'חיפה', 'דרום', 'באר שבע', 'תל אביב', 'ירושלים', ''],
      dtype=object)

In [102]:
demand.keys()

dict_keys(['Haifa', 'Tel Aviv', 'Beer Sheva', 'Hadera', 'Haifa Metronit', 'Jerusalem', 'Rail', 'Ashdod-Ashkelon'])

# update - no "Rail" in Mode Planned, but can contain Rail at the end.

In [103]:
# for i, row in gdf[gdf['Mode_Planned'].str.contains('Rail')].iterrows():
for i, row in gdf.iterrows():
  if row['area'] in ['חיפה','צפון']:
    gdf.iloc[i, gdf.columns.get_loc('TotalDemand')] = demand['Haifa'][demand['Haifa']['Node']==row['node']]['Boardings_Daily'].sum() + demand['Haifa'][demand['Haifa']['Node']==row['node']]['Alightings_Daily'].sum()
    gdf.iloc[i, gdf.columns.get_loc('TotalTransfers')] = demand['Haifa'][demand['Haifa']['Node']==row['node']]['TotalTransfers'].sum()
  elif row['area'] in ['תל אביב']:
    gdf.iloc[i, gdf.columns.get_loc('TotalDemand')] = demand['Tel Aviv'][demand['Tel Aviv']['Node']==row['node']]['Boardings_Daily'].sum() + demand['Tel Aviv'][demand['Tel Aviv']['Node']==row['node']]['Alightings_Daily'].sum()
    gdf.iloc[i, gdf.columns.get_loc('TotalTransfers')] = demand['Tel Aviv'][demand['Tel Aviv']['Node']==row['node']]['TotalTransfers'].sum()
  elif row['area'] in ['באר שבע','דרום']:
    gdf.iloc[i, gdf.columns.get_loc('TotalDemand')] = demand['Beer Sheva'][demand['Beer Sheva']['Node']==row['node']]['Boardings_Daily'].sum() + demand['Beer Sheva'][demand['Beer Sheva']['Node']==row['node']]['Alightings_Daily'].sum()
    gdf.iloc[i, gdf.columns.get_loc('TotalTransfers')] = demand['Beer Sheva'][demand['Beer Sheva']['Node']==row['node']]['TotalTransfers'].sum()
  elif row['area'] in ['ירושלים']:
    gdf.iloc[i, gdf.columns.get_loc('TotalDemand')] = demand['Jerusalem'][demand['Jerusalem']['Node']==row['node']]['Boardings_Daily'].sum() + demand['Jerusalem'][demand['Jerusalem']['Node']==row['node']]['Alightings_Daily'].sum()
    gdf.iloc[i, gdf.columns.get_loc('TotalTransfers')] = demand['Jerusalem'][demand['Jerusalem']['Node']==row['node']]['TotalTransfers'].sum()

for i, row in gdf[gdf['Mode_Planned'].str.contains('Rail')].iterrows():
  if row['node'] in [400820, 400841]:
    gdf.iloc[i, gdf.columns.get_loc('TotalDemand')] = demand['Ashdod-Ashkelon'][demand['Ashdod-Ashkelon']['Node']==row['node']]['Boardings_Daily'].sum() + demand['Ashdod-Ashkelon'][demand['Ashdod-Ashkelon']['Node']==row['node']]['Alightings_Daily'].sum()
    gdf.iloc[i, gdf.columns.get_loc('TotalTransfers')] = demand['Ashdod-Ashkelon'][demand['Ashdod-Ashkelon']['Node']==row['node']]['TotalTransfers'].sum()

/tmp/ipython-input-732918476.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '153.39311669' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  gdf.iloc[i, gdf.columns.get_loc('TotalTransfers')] = demand['Haifa'][demand['Haifa']['Node']==row['node']]['TotalTransfers'].sum()


In [104]:
# only kiryat gat and ashkelon train stations have demand taken from ashdod-ashkelon, all other were taken from tel-aviv

In [105]:
gdf[gdf['node']==511248]

,h3_index,node,Mode_Planned,Model,Line_Nunique,Line_Unique,geometry,group,address,area,location,TotalDemand,TotalTransfers
1128,8a3f4dc7052ffff,511248,LRT,Netanya,4,['LRT121' 'LRT122' 'LRT131' 'LRT132'],"POLYGON ((34.84088769100656 32.21510746276794,...",774,"איילון צפון, מועצה אזורית חוף השרון, נפת השרון...",תל אביב,טבעת חיצונית,253.9,0.0


# Update Shefaim LRT stop

In [106]:
gdf.loc[gdf['node']==511248,'TotalDemand'] = 255.3

In [107]:
gdf[gdf['node']==511248]

,h3_index,node,Mode_Planned,Model,Line_Nunique,Line_Unique,geometry,group,address,area,location,TotalDemand,TotalTransfers
1128,8a3f4dc7052ffff,511248,LRT,Netanya,4,['LRT121' 'LRT122' 'LRT131' 'LRT132'],"POLYGON ((34.84088769100656 32.21510746276794,...",774,"איילון צפון, מועצה אזורית חוף השרון, נפת השרון...",תל אביב,טבעת חיצונית,255.3,0.0


In [108]:
gdf.to_csv('/content/drive/MyDrive/Hubs/groups_hubs_with_demand_29102025.csv', encoding='utf-8', index=False)

In [109]:
gdf = gpd.read_file('/content/drive/MyDrive/Hubs/groups_hubs_with_demand_29102025.csv', encoding='utf-8')

In [110]:
gdf['node'] = gdf['node'].astype('int64')

In [ ]:
# add 2050 employees & population in proximity to the h3_index
# first - group the hub list
# create centroid for the group, not the one h3_index.
# create 3 rings of influence - 0-600 meter (core), 600-1000 meter (primary) and 1000-1200 (extended)
# the whole 1.2 km is an influence area
# the scoring is weighted by the ring (1, 0.6 and 0.3)
# scoring should be normalized before weights


In [111]:
# update Haifa Metronit using Haifa Metronit demand data

# Ensure 'Node' column in demand['Haifa Metronit'] is numeric
demand['Haifa Metronit']['Node'] = pd.to_numeric(demand['Haifa Metronit']['Node'], errors='coerce')

# Select relevant columns from demand['Haifa Metronit']
haifa_metronit_demand = demand['Haifa Metronit'][['Node', 'Boardings_Daily', 'Alightings_Daily', 'TotalTransfers']]

# Aggregate haifa_metronit_demand by Node, summing the demand and transfers
haifa_metronit_demand_agg = haifa_metronit_demand.groupby('Node').agg(
    Boardings_Haifa_Metronit_sum=('Boardings_Daily', 'sum'),
    Alightings_Haifa_Metronit_sum=('Alightings_Daily', 'sum'),
    TotalTransfers_Haifa_Metronit_sum=('TotalTransfers', 'sum')
).reset_index() # Reset index to make 'Node' a column again

# Rename 'Node' to 'node' for matching with gdf
haifa_metronit_demand_agg.rename(columns={'Node': 'node'}, inplace=True)

# Set 'node' as index in haifa_metronit_demand_agg for efficient lookups
haifa_metronit_demand_agg.set_index('node', inplace=True)

# Update TotalDemand and TotalTransfers in gdf using data from haifa_metronit_demand_agg
# Iterate through gdf and update rows where node matches in haifa_metronit_demand_agg
for index, row in gdf.iterrows():
    if row['node'] in haifa_metronit_demand_agg.index:
        # Get corresponding demand data from haifa_metronit_demand_agg
        haifa_metronit_data = haifa_metronit_demand_agg.loc[row['node']]

        # Update TotalDemand and TotalTransfers, adding to existing values
        gdf.loc[index, 'TotalDemand'] = gdf.loc[index, 'TotalDemand'] + haifa_metronit_data['Boardings_Haifa_Metronit_sum'] + haifa_metronit_data['Alightings_Haifa_Metronit_sum']
        gdf.loc[index, 'TotalTransfers'] = gdf.loc[index, 'TotalTransfers'] + haifa_metronit_data['TotalTransfers_Haifa_Metronit_sum']

# Display the head of gdf to check the updated columns
display(gdf.head())

,h3_index,node,Mode_Planned,Model,Line_Nunique,Line_Unique,geometry,group,address,area,location,TotalDemand,TotalTransfers
0,8a2da4100a27fff,31655,Interurban Rail,National,2,['rail_1_1' 'rail_1_2'],"POLYGON ((35.58265301880704 33.2100839260914, ...",0,"מנחם בגין/נורית, שדרות מנחם בגין, הורדים, קרית...",צפון,צפון,3037.68534095,153.39311669
1,8a2da4c22257fff,36963,BRT,Haifa,1,['brt022'],"POLYGON ((35.32671223620188 32.92554871232436,...",1,"מיי סנטר, מעלה כמון, אזור תעשייה כרמיאל, אזור ...",צפון,צפון,920.5066456,920.5066456
2,8a2da4c22627fff,36959,BRT,Haifa,1,['brt021'],POLYGON ((35.31703160566268 32.922357121183495...,2,"החרושת/חשמל, החרושת, אזור תעשייה כרמיאל, מרכז ...",צפון,צפון,0.56331008,0.56331008
3,8a2da4c2275ffff,36961,BRT,Haifa,2,['brt021' 'brt022'],"POLYGON ((35.31899096754 32.923530411401764, 3...",2,"החרושת, אזור תעשייה כרמיאל, אזור תעשׂייה, כרמיא...",צפון,צפון,609.6958255999999,119.20414412000001
4,8a2da4c24987fff,36924,BRT,Haifa,4,['brt011' 'brt012' 'brt021' 'brt022'],"POLYGON ((35.34684915390923 32.86202464664849,...",3,"طريق الجليل, عرابة, נפת עכו, מחוז הצפון, 20176...",צפון,צפון,322.391024,322.391024


In [112]:
gdf[gdf['h3_index']=='8a3f4dca140ffff']

,h3_index,node,Mode_Planned,Model,Line_Nunique,Line_Unique,geometry,group,address,area,location,TotalDemand,TotalTransfers
1133,8a3f4dca140ffff,511037,LRT,Tel Aviv,2,['Red-N' 'Red2-N'],POLYGON ((34.75773243623256 31.989616192921012...,779,"איילון דרום, אזור תעשייה חדש, נווה דקלים, ראשו...",תל אביב,טבעת תיכונה,2916.87,965.9


# Update specific nodes demand from National Model


*   Netanya
*   Netanya Sapir
*   Beit Yehoshua
*   Moshe Dayan (Rishon)
*   Modiin Merkaz
*   Modiin West



### Moshe Dayan

In [113]:
gdf.loc[gdf['node']==400424, 'TotalDemand'] = 64985
gdf.loc[gdf['node']==400424, 'TotalTransfers'] = 43032

In [114]:
gdf[gdf['group']=='777']

,h3_index,node,Mode_Planned,Model,Line_Nunique,Line_Unique,geometry,group,address,area,location,TotalDemand,TotalTransfers
1131,8a3f4dca0877fff,516020,LRT,Tel Aviv,4,['ART1-1' 'ART1-2' 'ART2-1' 'ART2-2'],POLYGON ((34.77526307358566 31.970503038257228...,777,"לוי אשכול, קריית כרמים, ראשון לציון, נפת רחובו...",תל אביב,טבעת תיכונה,16191.939999999999,250.76


### Netanya

In [115]:
gdf[gdf['node']==400020]

,h3_index,node,Mode_Planned,Model,Line_Nunique,Line_Unique,geometry,group,address,area,location,TotalDemand,TotalTransfers
1064,8a3f4dc610cffff,400020,Interurban Rail,National,8,['rail_3_1' 'rail_3_2' 'rail_4_1' 'rail_4_2' '...,"POLYGON ((34.86966980630241 32.31835451598878,...",738,"הרכבת, רמת אפרים, נתניה, נפת השרון, מחוז המרכז...",תל אביב,טבעת חיצונית,19126.69,15573.920000000002
1065,8a3f4dc610cffff,400020,Suburban Rail,National,6,['rail_11_1' 'rail_11_2' 'rail_12_1' 'rail_12_...,"POLYGON ((34.86966980630241 32.31835451598878,...",738,"הרכבת, רמת אפרים, נתניה, נפת השרון, מחוז המרכז...",תל אביב,טבעת חיצונית,19126.69,15573.920000000002


Because we don't have distinguish in the national model, regarding which mode_planned has the TotalDemand, we will set to one of the mode_planned, as long as they are the same node.

In [116]:
gdf.loc[gdf.index==1057,:]

,h3_index,node,Mode_Planned,Model,Line_Nunique,Line_Unique,geometry,group,address,area,location,TotalDemand,TotalTransfers
1057,8a3f4dc60657fff,511243,BRT,Netanya,1,['LRT141'],POLYGON ((34.866372549223634 32.29050588526704...,734,"המחקר, אזור התעשיה פולג, רמת ידין, נתניה, נפת ...",תל אביב,טבעת חיצונית,3235.85,22.15


In [117]:
gdf.loc[gdf.index==1057, 'TotalDemand'] = 108409
gdf.loc[gdf.index==1057, 'TotalTransfers'] = 84490

# update both, but only 1 will be taken, so there will be no mistakes
gdf.loc[gdf.index==1058, 'TotalDemand'] = 108409
gdf.loc[gdf.index==1058, 'TotalTransfers'] = 84490

In [118]:
gdf[gdf['group']=='736']

,h3_index,node,Mode_Planned,Model,Line_Nunique,Line_Unique,geometry,group,address,area,location,TotalDemand,TotalTransfers
1061,8a3f4dc6071ffff,511222,LRT,Netanya,4,['LRT121' 'LRT122' 'LRT131' 'LRT132'],"POLYGON ((34.86935058312575 32.28620830346105,...",736,"הארי טרומן, קריית נורדאו, נתניה, נפת השרון, מח...",תל אביב,טבעת חיצונית,11165.05,8769.82


### Netanya Sapir

In [119]:
gdf[gdf['node']==400021]

,h3_index,node,Mode_Planned,Model,Line_Nunique,Line_Unique,geometry,group,address,area,location,TotalDemand,TotalTransfers
1098,8a3f4dc62ba7fff,400021,Suburban Rail,National,4,['rail_11_1' 'rail_11_2' 'rail_14_1' 'rail_14_2'],POLYGON ((34.864303798088294 32.27460860271364...,750,"ארקפה, בני גאון, אזור התעשיה פולג, קריית נורדא...",תל אביב,טבעת חיצונית,7074.07,4375.469999999999


In [120]:
gdf[gdf['group']=='748']

,h3_index,node,Mode_Planned,Model,Line_Nunique,Line_Unique,geometry,group,address,area,location,TotalDemand,TotalTransfers
1084,8a3f4dc62077fff,511228,BRT,Netanya,1,['LRT141'],"POLYGON ((34.8578157731143 32.2725281167698, 3...",748,"שדרות גיבורי ישראל, אזור התעשיה פולג, קריית נו...",תל אביב,טבעת חיצונית,12.53001784,12.53001784
1085,8a3f4dc62077fff,511228,LRT,Netanya,1,['LRT142'],"POLYGON ((34.8578157731143 32.2725281167698, 3...",748,"שדרות גיבורי ישראל, אזור התעשיה פולג, קריית נו...",תל אביב,טבעת חיצונית,12.53001784,12.53001784
1086,8a3f4dc6215ffff,511227,BRT,Netanya,1,['LRT141'],POLYGON ((34.859086753292296 32.27241037849170...,748,"איקאה, מפי, אזור התעשיה פולג, קריית נורדאו, נת...",תל אביב,טבעת חיצונית,2197.66,187.42
1087,8a3f4dc6215ffff,511227,LRT,Netanya,1,['LRT142'],POLYGON ((34.859086753292296 32.27241037849170...,748,"איקאה, מפי, אזור התעשיה פולג, קריית נורדאו, נת...",תל אביב,טבעת חיצונית,2197.66,187.42
1092,8a3f4dc62a0ffff,511224,BRT,Netanya,1,['LRT141'],POLYGON ((34.862028093123996 32.28018223703595...,748,"דומינו'ס פיצה, שדרות גיבורי ישראל, אזור התעשיה...",תל אביב,טבעת חיצונית,3465.95,3169.8
1093,8a3f4dc62a0ffff,511224,LRT,Netanya,1,['LRT142'],POLYGON ((34.862028093123996 32.28018223703595...,748,"דומינו'ס פיצה, שדרות גיבורי ישראל, אזור התעשיה...",תל אביב,טבעת חיצונית,3465.95,3169.8
1094,8a3f4dc62a17fff,511225,BRT,Netanya,1,['LRT141'],"POLYGON ((34.86189502068818 32.27751314434144,...",748,"KSP, האומנות, אזור התעשיה פולג, קריית נורדאו, ...",תל אביב,טבעת חיצונית,5159.209999999999,22.830222999999997
1095,8a3f4dc62a17fff,511225,LRT,Netanya,1,['LRT142'],"POLYGON ((34.86189502068818 32.27751314434144,...",748,"KSP, האומנות, אזור התעשיה פולג, קריית נורדאו, ...",תל אביב,טבעת חיצונית,5159.209999999999,22.830222999999997
1096,8a3f4dc62ab7fff,511226,BRT,Netanya,1,['LRT141'],POLYGON ((34.86049092486111 32.274961738263904...,748,"5א, מפי, אזור התעשיה פולג, קריית נורדאו, נתניה...",תל אביב,טבעת חיצונית,39.520187459999995,39.520187459999995
1097,8a3f4dc62ab7fff,511226,LRT,Netanya,1,['LRT142'],POLYGON ((34.86049092486111 32.274961738263904...,748,"5א, מפי, אזור התעשיה פולג, קריית נורדאו, נתניה...",תל אביב,טבעת חיצונית,39.520187459999995,39.520187459999995


In [121]:
gdf.loc[gdf['node']==400021, 'TotalDemand'] = 23083
gdf.loc[gdf['node']==400021, 'TotalTransfers'] = 10140

### Beit Yehoshua

In [122]:
gdf[gdf['node']==400030]

,h3_index,node,Mode_Planned,Model,Line_Nunique,Line_Unique,geometry,group,address,area,location,TotalDemand,TotalTransfers
1101,8a3f4dc62ce7fff,400030,Suburban Rail,National,4,['rail_11_1' 'rail_11_2' 'rail_14_1' 'rail_14_2'],POLYGON ((34.859824340410164 32.26161601157606...,751,"ספיר, מועצה אזורית חוף השרון, נפת השרון, מחוז ...",תל אביב,טבעת חיצונית,4842.8099999999995,311.730129


In [123]:
gdf[gdf['group']=='749']

,h3_index,node,Mode_Planned,Model,Line_Nunique,Line_Unique,geometry,group,address,area,location,TotalDemand,TotalTransfers
1088,8a3f4dc62287fff,511230,BRT,Netanya,1,['LRT141'],"POLYGON ((34.85102510872771 32.27717886282845,...",749,"בן גוריון, קריית נורדאו, נתניה, נפת השרון, מחו...",תל אביב,טבעת חיצונית,5121.59,1593.59
1089,8a3f4dc62287fff,511230,LRT,Netanya,1,['LRT142'],"POLYGON ((34.85102510872771 32.27717886282845,...",749,"בן גוריון, קריית נורדאו, נתניה, נפת השרון, מחו...",תל אביב,טבעת חיצונית,5121.59,1593.59
1090,8a3f4dc6239ffff,511229,BRT,Netanya,1,['LRT141'],"POLYGON ((34.85286502194533 32.2756679253614, ...",749,"גולדה מאיר, נאות גולדה, נתניה, נפת השרון, מחוז...",תל אביב,טבעת חיצונית,0.0,0.0
1091,8a3f4dc6239ffff,511229,LRT,Netanya,1,['LRT142'],"POLYGON ((34.85286502194533 32.2756679253614, ...",749,"גולדה מאיר, נאות גולדה, נתניה, נפת השרון, מחוז...",תל אביב,טבעת חיצונית,0.0,0.0


Because we have lrt here that is in the same group as the rail,<br>
we will adjust the total demand for the lrt node as well.

In [124]:
# Rail
gdf.loc[gdf['node']==400030, 'TotalDemand'] = 14518
gdf.loc[gdf['node']==400030, 'TotalTransfers'] = 6101

# LRT
gdf.loc[gdf['node']==511246, 'TotalDemand'] = 13601
gdf.loc[gdf['node']==511246, 'TotalTransfers'] = 6101


### Modiin Merkaz

In [125]:
gdf[gdf['node']==400470]

,h3_index,node,Mode_Planned,Model,Line_Nunique,Line_Unique,geometry,group,address,area,location,TotalDemand,TotalTransfers
421,8a3f4ca4d8effff,400470,Interurban Rail,National,2,['rail_1_1' 'rail_1_2'],POLYGON ((35.00609896305314 31.901280226811465...,311,"ערער, הפרחים, מודיעין-מכבים-רעות, נפת רמלה, מח...",תל אביב,טבעת חיצונית,13395.68,3541.2699999999995
422,8a3f4ca4d8effff,400470,Suburban Rail,National,6,['rail_19_1' 'rail_19_2' 'rail_20_1' 'rail_20_...,POLYGON ((35.00609896305314 31.901280226811465...,311,"ערער, הפרחים, מודיעין-מכבים-רעות, נפת רמלה, מח...",תל אביב,טבעת חיצונית,13395.68,3541.2699999999995


In [126]:
gdf[gdf['group']=='311']

,h3_index,node,Mode_Planned,Model,Line_Nunique,Line_Unique,geometry,group,address,area,location,TotalDemand,TotalTransfers
421,8a3f4ca4d8effff,400470,Interurban Rail,National,2,['rail_1_1' 'rail_1_2'],POLYGON ((35.00609896305314 31.901280226811465...,311,"ערער, הפרחים, מודיעין-מכבים-רעות, נפת רמלה, מח...",תל אביב,טבעת חיצונית,13395.68,3541.2699999999995
422,8a3f4ca4d8effff,400470,Suburban Rail,National,6,['rail_19_1' 'rail_19_2' 'rail_20_1' 'rail_20_...,POLYGON ((35.00609896305314 31.901280226811465...,311,"ערער, הפרחים, מודיעין-מכבים-רעות, נפת רמלה, מח...",תל אביב,טבעת חיצונית,13395.68,3541.2699999999995


Because we don't have distinguish in the national model, regarding which mode_planned has the TotalDemand, we will set to one of the mode_planned, as long as they are the same node.

In [127]:
gdf.loc[gdf.index==421, 'TotalDemand'] = 40628
gdf.loc[gdf.index==421, 'TotalTransfers'] = 0

gdf.loc[gdf.index==422, 'TotalDemand'] = 40628
gdf.loc[gdf.index==422, 'TotalTransfers'] = 0

### Modiin West

In [128]:
gdf[gdf['node']==400460]

,h3_index,node,Mode_Planned,Model,Line_Nunique,Line_Unique,geometry,group,address,area,location,TotalDemand,TotalTransfers
419,8a3f4ca4b11ffff,400460,Interurban Rail,National,2,['rail_1_1' 'rail_1_2'],POLYGON ((34.960910680536614 31.89220354012376...,310,"431, מודיעין-מכבים-רעות, נפת רמלה, מחוז המרכז,...",תל אביב,טבעת חיצונית,62552.92,53336.99
420,8a3f4ca4b11ffff,400460,Suburban Rail,National,6,['rail_19_1' 'rail_19_2' 'rail_20_1' 'rail_20_...,POLYGON ((34.960910680536614 31.89220354012376...,310,"431, מודיעין-מכבים-רעות, נפת רמלה, מחוז המרכז,...",תל אביב,טבעת חיצונית,62552.92,53336.99


In [129]:
gdf[gdf['group']=='310']

,h3_index,node,Mode_Planned,Model,Line_Nunique,Line_Unique,geometry,group,address,area,location,TotalDemand,TotalTransfers
419,8a3f4ca4b11ffff,400460,Interurban Rail,National,2,['rail_1_1' 'rail_1_2'],POLYGON ((34.960910680536614 31.89220354012376...,310,"431, מודיעין-מכבים-רעות, נפת רמלה, מחוז המרכז,...",תל אביב,טבעת חיצונית,62552.92,53336.99
420,8a3f4ca4b11ffff,400460,Suburban Rail,National,6,['rail_19_1' 'rail_19_2' 'rail_20_1' 'rail_20_...,POLYGON ((34.960910680536614 31.89220354012376...,310,"431, מודיעין-מכבים-רעות, נפת רמלה, מחוז המרכז,...",תל אביב,טבעת חיצונית,62552.92,53336.99


Because we don't have distinguish in the national model, regarding which mode_planned has the TotalDemand, we will set to one of the mode_planned, as long as they are the same node.

In [130]:
gdf.loc[gdf.index==419, 'TotalDemand'] = 41000
gdf.loc[gdf.index==419, 'TotalTransfers'] = 12133

gdf.loc[gdf.index==420, 'TotalDemand'] = 41000
gdf.loc[gdf.index==420, 'TotalTransfers'] = 12133

In [131]:
gdf.to_csv('/content/drive/MyDrive/Hubs/groups_hubs_with_demand_29102025.csv', encoding='utf-8', index=False)

In [132]:
gdf = gpd.read_file('/content/drive/MyDrive/Hubs/groups_hubs_with_demand_29102025.csv', encoding='utf-8')

In [133]:
gdf.head(1)

,h3_index,node,Mode_Planned,Model,Line_Nunique,Line_Unique,geometry,group,address,area,location,TotalDemand,TotalTransfers
0,8a2da4100a27fff,31655,Interurban Rail,National,2,['rail_1_1' 'rail_1_2'],"POLYGON ((35.58265301880704 33.2100839260914, ...",0,"מנחם בגין/נורית, שדרות מנחם בגין, הורדים, קרית...",צפון,צפון,3037.68534095,153.39311669


In [134]:
gdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1505 entries, 0 to 1504
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   h3_index        1505 non-null   object
 1   node            1505 non-null   object
 2   Mode_Planned    1505 non-null   object
 3   Model           1505 non-null   object
 4   Line_Nunique    1505 non-null   object
 5   Line_Unique     1505 non-null   object
 6   geometry        1505 non-null   object
 7   group           1505 non-null   object
 8   address         1505 non-null   object
 9   area            1505 non-null   object
 10  location        1505 non-null   object
 11  TotalDemand     1505 non-null   object
 12  TotalTransfers  1505 non-null   object
dtypes: object(13)
memory usage: 153.0+ KB


In [135]:
gdf['Line_Nunique'] = pd.to_numeric(gdf['Line_Nunique'], errors='coerce').fillna(0)
gdf['TotalDemand'] = pd.to_numeric(gdf['TotalDemand'], errors='coerce').fillna(0)
gdf['TotalTransfers'] = pd.to_numeric(gdf['TotalTransfers'], errors='coerce').fillna(0)

In [136]:
from shapely import wkt
import geopandas as gpd

# Convert the string representations of geometries to shapely objects
gdf['geometry'] = gdf['geometry'].apply(wkt.loads)

gdf = gpd.GeoDataFrame(gdf, geometry='geometry', crs=crs_il)

In [137]:
gdf.crs

<Projected CRS: EPSG:2039>
Name: Israel 1993 / Israeli TM Grid
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: Israel - onshore; Palestine Territory - onshore.
- bounds: (34.17, 29.45, 35.69, 33.28)
Coordinate Operation:
- name: Israeli TM
- method: Transverse Mercator
Datum: Israel 1993
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

In [ ]:
# # Ensure 'geometry' column contains valid geometry objects
# gdf['geometry'] = gdf['geometry'].apply(wkt.loads)

# # Ensure gdf is a GeoDataFrame before dissolving
# # Assuming 'geometry' is the geometry column and crs_il is the correct CRS
# gdf = gpd.GeoDataFrame(gdf, geometry='geometry', crs=crs_il)


# # Group the gdf by the 'group' column and dissolve geometries using dissolve()
# grouped_hubs = gdf.dissolve(by='group', aggfunc={
#     'h3_index': 'first',  # Keep the first h3_index for reference
#     'node': lambda x: list(x), # Aggregate nodes into a list
#     'Mode_Planned': lambda x: list(x.unique()), # Aggregate unique Mode_Planned into a list
#     'Model': lambda x: list(x.unique()), # Aggregate unique Models into a list
#     'Line_Nunique': 'sum',  # Sum the unique line counts within the group
#     'Line_Unique': lambda x: list(set([item for sublist in x for item in sublist])), # Aggregate unique lines into a list
#     'address': lambda x: x.iloc[0] if not x.isnull().all() else None, # Keep the first non-null address
#     'area': lambda x: list(x.unique()), # Aggregate unique areas into a list
#     'location': lambda x: list(x.unique()), # Aggregate unique locations into a list
#     'TotalDemand': 'sum', # Sum the total demand for the group
#     'TotalTransfers': 'sum' # Sum the total transfers for the group
# }).reset_index() # Reset index to make 'group' a column


# # Convert the result to a GeoDataFrame (dissolve already returns a GeoDataFrame, but good practice to be explicit)
# grouped_hubs = gpd.GeoDataFrame(grouped_hubs, geometry='geometry', crs=gdf.crs)


# # Create centroid for each group's geometry
# # Ensure the GeoDataFrame is in a projected CRS before calculating centroids for accurate results
# if grouped_hubs.crs.is_geographic:
#     grouped_hubs = grouped_hubs.to_crs(epsg=crs_il) # Reproject to a projected CRS for Israel

# grouped_hubs['centroid'] = grouped_hubs.geometry.centroid

# # Display the head of the new grouped_hubs GeoDataFrame
# display(grouped_hubs.head())

,group,geometry,h3_index,node,Mode_Planned,Model,Line_Nunique,Line_Unique,address,area,location,TotalDemand,TotalTransfers,centroid
0,0,"POLYGON ((35.583 33.21, 35.582 33.21, 35.581 3...",8a2da4100a27fff,[31655],[Interurban Rail],[national],2,"[a, ], 2, , i, l, _, 1, ', r, []","מנחם בגין/נורית, שדרות מנחם בגין, הורדים, קרית...",[צפון],[צפון],0.000000,0.000000,POINT (35.582 33.211)
1,1,"POLYGON ((35.327 32.926, 35.326 32.925, 35.325...",8a2da4c22257fff,[36963],[BRT],[haifa],1,"[], 2, t, 0, b, ', r, []","מיי סנטר, מעלה כמון, אזור תעשייה כרמיאל, כרמיא...",[צפון],[צפון],920.506646,920.506646,POINT (35.326 32.926)
2,10,"POLYGON ((35.305 32.917, 35.304 32.916, 35.303...",8a2da4c31877fff,[36950],[BRT],[haifa],2,"[], 2, t, , 0, 1, b, ', r, []","הגליל א׳, הגליל, שיכון המייסדים, כרמיאל, נפת ע...",[צפון],[צפון],25.996522,25.996522,POINT (35.304 32.917)
3,100,"POLYGON ((34.793 31.258, 34.792 31.257, 34.791...",8a3f4c02b2a7fff,[467909],[LRT],[beer sheva],2,"[., a, ], , d, S, _, -, B, [, A, ', 4, r, 1]","Caldo, 47, מצדה, שכונה ב', שכונה ב, באר שבע, נ...",[באר שבע],[גלעין],1151.846630,0.000000,POINT (34.792 31.258)
4,1000,"POLYGON ((35.241 31.819, 35.24 31.819, 35.239 ...",8a3f6b79ed2ffff,[51004],[LRT],[jerusalem],4,"[T, ], 2, , L, R, ', 5, 1, []","אלוף יקותיאל אדם, פסגת זאב מערב, פסגת זאב, ירו...",[ירושלים],[גלעין],30027.239744,0.000000,POINT (35.24 31.82)


In [138]:
combined[combined['Node']==511253]

,Node,Boardings_Daily,Alightings_Daily,TotalTransfers,Model
25837,511253,595.53,555.46,1031.85,Tel Aviv


In [139]:
gdf[gdf['group']=='558']

,h3_index,node,Mode_Planned,Model,Line_Nunique,Line_Unique,geometry,group,address,area,location,TotalDemand,TotalTransfers
780,8a3f4dc08947fff,511254,LRT,Netanya,4,['LRT121' 'LRT122' 'LRT131' 'LRT132'],"POLYGON ((34.819 32.164, 34.818 32.163, 34.818...",558,"הרכבת, שכונת גליל ים, הרצליה ב', הרצליה, נפת ת...",תל אביב,טבעת פנימית,19546.40,18012.12
781,8a3f4dc0895ffff,400060,Interurban Rail,National,8,['rail_3_1' 'rail_3_2' 'rail_4_1' 'rail_4_2' '...,"POLYGON ((34.818 32.164, 34.817 32.163, 34.817...",558,"מחלף שבעת הכוכבים, איילון צפון, שכונת גליל ים,...",תל אביב,טבעת פנימית,28743.58,25195.63
782,8a3f4dc0895ffff,400060,Suburban Rail,National,14,['rail_10_1' 'rail_10_2' 'rail_11_1' 'rail_11_...,"POLYGON ((34.818 32.164, 34.817 32.163, 34.817...",558,"מחלף שבעת הכוכבים, איילון צפון, שכונת גליל ים,...",תל אביב,טבעת פנימית,28743.58,25195.63
783,8a3f4dc08967fff,525318,BRT,Tel Aviv,2,['Hz-KS1' 'Hz-KS2'],"POLYGON ((34.82 32.163, 34.82 32.163, 34.819 3...",558,"בן ציון מיכאלי, הרצליה ב', הרצליה, נפת תל אביב...",תל אביב,טבעת פנימית,11633.08,11446.51
784,8a3f4dc0896ffff,511253,LRT,Netanya,4,['LRT121' 'LRT122' 'LRT131' 'LRT132'],"POLYGON ((34.82 32.165, 34.819 32.164, 34.819 ...",558,"איילון צפון, שכונת גליל ים, הרצליה ב', הרצליה,...",תל אביב,טבעת פנימית,1150.99,1031.85


# This code has a problem with double summarizing same nodes

In [ ]:
# # Ensure 'geometry' column contains valid geometry objects
# # gdf['geometry'] = gdf['geometry'].apply(wkt.loads) # This should be done after loading from CSV if needed

# # Ensure gdf is a GeoDataFrame before dissolving
# # Assuming 'geometry' is the geometry column and crs_il is the correct CRS
# # This line was already added in cell 9cd93278 and should not be repeated here unless gdf is reloaded as a DataFrame
# # gdf = gpd.GeoDataFrame(gdf, geometry='geometry', crs=crs_il)

# # Convert 'group' column to integer type for consistent merging and aggregation
# gdf['group'] = pd.to_numeric(gdf['group'], errors='coerce').fillna(-1).astype(int)

# # Aggregate TotalDemand and TotalTransfers per unique node within each group *before* dissolving
# node_demand_agg = gdf.groupby(['group', 'node']).agg({
#     'TotalDemand': 'sum',
#     'TotalTransfers': 'sum'
# }).reset_index()

# # Ensure 'group' column in node_demand_agg is also integer type
# node_demand_agg['group'] = node_demand_agg['group'].astype(int)


# # Create a dictionary mapping group IDs to their total demand and transfers for easier lookup during dissolve
# group_demand_lookup = node_demand_agg.groupby('group').agg({
#     'TotalDemand': 'sum',
#     'TotalTransfers': 'sum'
# }).to_dict('index')

# # Group the gdf by the 'group' column and dissolve geometries using dissolve()
# # Use custom aggregation for TotalDemand and TotalTransfers by looking up values in group_demand_lookup
# grouped_hubs = gdf.dissolve(by='group', aggfunc={
#     'h3_index': list,  # Keep the first h3_index for reference
#     'node': lambda x: list(x), # Aggregate nodes into a list
#     'Mode_Planned': lambda x: list(x.unique()), # Aggregate unique Mode_Planned into a list
#     'Model': lambda x: list(x.unique()), # Aggregate unique Models into a list
#     # Modified Line_Unique aggregation to return a flat list of unique lines
#     'Line_Unique': lambda x: list(set([item for sublist in x for item in (sublist if isinstance(sublist, (list, np.ndarray)) else [sublist])])),
#     'address': lambda x: x.iloc[0] if not x.isnull().all() else None, # Keep the first non-null address
#     'area': lambda x: list(x.unique()), # Aggregate unique areas into a list
#     'location': lambda x: list(x.unique()), # Aggregate unique locations into a list
#     # Custom aggregation for TotalDemand and TotalTransfers using the lookup dictionary
#     'TotalDemand': lambda x: group_demand_lookup.get(x.name, {}).get('TotalDemand', 0),
#     'TotalTransfers': lambda x: group_demand_lookup.get(x.name, {}).get('TotalTransfers', 0)
# }).reset_index() # Reset index to make 'group' a column


# # Convert the result to a GeoDataFrame (dissolve already returns a GeoDataFrame, but good practice to be explicit)
# grouped_hubs = gpd.GeoDataFrame(grouped_hubs, geometry='geometry', crs=gdf.crs)


# # Calculate the number of unique lines per group based on the aggregated 'Line_Unique' list
# grouped_hubs['Total_Unique_Lines'] = grouped_hubs['Line_Unique'].apply(lambda x: len(x) if isinstance(x, list) else 0)


# # Calculate sum of Line_Nunique per Mode_Planned per group
# # This part remains useful for seeing the breakdown by mode within the group
# line_nunique_by_mode = gdf.groupby(['group', 'Mode_Planned'])['Line_Nunique'].sum().unstack(fill_value=0)

# # Rename columns to desired format (e.g., 'BRT_Line_Nunique')
# line_nunique_by_mode.columns = [f'{col} Lines' for col in line_nunique_by_mode.columns]

# # Merge the new Line_Nunique columns back into grouped_hubs
# grouped_hubs = grouped_hubs.merge(line_nunique_by_mode, left_on='group', right_index=True, how='left')


# # Create centroid for each group's geometry
# # Ensure the GeoDataFrame is in a projected CRS before calculating centroids for accurate results
# if grouped_hubs.crs.is_geographic:
#     grouped_hubs = grouped_hubs.to_crs(epsg=crs_il) # Reproject to a projected CRS for Israel

# grouped_hubs['centroid'] = grouped_hubs.geometry.centroid

# # Display the head of the new grouped_hubs GeoDataFrame
# display(grouped_hubs.head())

# This code try to get around the problem


In [140]:
# Ensure 'geometry' column contains valid geometry objects
# gdf['geometry'] = gdf['geometry'].apply(wkt.loads) # This should be done after loading from CSV if needed

# Ensure gdf is a GeoDataFrame before dissolving
# Assuming 'geometry' is the geometry column and crs_il is the correct CRS
# This line was already added in cell 9cd93278 and should not be repeated here unless gdf is reloaded as a DataFrame
# gdf = gpd.GeoDataFrame(gdf, geometry='geometry', crs=crs_il)

# Convert 'group' column to integer type for consistent merging and aggregation
gdf['group'] = pd.to_numeric(gdf['group'], errors='coerce').fillna(-1).astype(int)

# First, for each unique combination of group+node, take only the first occurrence
# This prevents double-counting the same node within the same group
gdf_unique_nodes = gdf.drop_duplicates(subset=['group', 'node'], keep='first')

# Now aggregate by group, summing across all unique nodes within each group
# This will sum TotalDemand from node1 + node2 + node3, etc. within each group
# but won't double-count if the same node appears multiple times
group_demand_totals = gdf_unique_nodes.groupby('group').agg({
    'TotalDemand': 'sum',
    'TotalTransfers': 'sum'
}).reset_index()

# Group the original gdf by the 'group' column and dissolve geometries using dissolve()
grouped_hubs = gdf.dissolve(by='group', aggfunc={
    'h3_index': list,  # Keep the first h3_index for reference
    'node': lambda x: list(x), # Aggregate nodes into a list
    'Mode_Planned': lambda x: list(x.unique()), # Aggregate unique Mode_Planned into a list
    'Model': lambda x: list(x.unique()), # Aggregate unique Models into a list
    # Modified Line_Unique aggregation to return a flat list of unique lines
    'Line_Unique': lambda x: list(set([item for sublist in x for item in (sublist if isinstance(sublist, (list, np.ndarray)) else [sublist])])),
    'address': lambda x: x.iloc[0] if not x.isnull().all() else None, # Keep the first non-null address
    'area': lambda x: list(x.unique()), # Aggregate unique areas into a list
    'location': lambda x: list(x.unique()) # Aggregate unique locations into a list
}).reset_index() # Reset index to make 'group' a column

# Convert the result to a GeoDataFrame (dissolve already returns a GeoDataFrame, but good practice to be explicit)
grouped_hubs = gpd.GeoDataFrame(grouped_hubs, geometry='geometry', crs=gdf.crs)

# Merge the pre-calculated demand totals back into grouped_hubs
grouped_hubs = grouped_hubs.merge(group_demand_totals, on='group', how='left')

# Fill any NaN values with 0 (in case some groups don't have demand data)
grouped_hubs['TotalDemand'] = grouped_hubs['TotalDemand'].fillna(0)
grouped_hubs['TotalTransfers'] = grouped_hubs['TotalTransfers'].fillna(0)

# Calculate the number of unique lines per group based on the aggregated 'Line_Unique' list
grouped_hubs['Total_Unique_Lines'] = grouped_hubs['Line_Unique'].apply(lambda x: len(x) if isinstance(x, list) else 0)

# Calculate sum of Line_Nunique per Mode_Planned per group
# This part remains useful for seeing the breakdown by mode within the group
line_nunique_by_mode = gdf.groupby(['group', 'Mode_Planned'])['Line_Nunique'].sum().unstack(fill_value=0)

# Rename columns to desired format (e.g., 'BRT_Line_Nunique')
line_nunique_by_mode.columns = [f'{col} Lines' for col in line_nunique_by_mode.columns]

# Merge the new Line_Nunique columns back into grouped_hubs
grouped_hubs = grouped_hubs.merge(line_nunique_by_mode, left_on='group', right_index=True, how='left')

# Create centroid for each group's geometry
# Ensure the GeoDataFrame is in a projected CRS before calculating centroids for accurate results
if grouped_hubs.crs.is_geographic:
    grouped_hubs = grouped_hubs.to_crs(epsg=crs_il) # Reproject to a projected CRS for Israel

grouped_hubs['centroid'] = grouped_hubs.geometry.centroid

# Display the head of the new grouped_hubs GeoDataFrame
display(grouped_hubs.head())

# Debug: Check if demand totals were calculated correctly
print("Group demand totals:")
print(group_demand_totals.head())
print(f"\nTotal groups with demand data: {len(group_demand_totals)}")
print(f"Groups with non-zero TotalDemand: {(group_demand_totals['TotalDemand'] > 0).sum()}")
print(f"Groups with non-zero TotalTransfers: {(group_demand_totals['TotalTransfers'] > 0).sum()}")

# Additional debug: Check unique nodes per group
print("\nUnique nodes per group (first 10 groups):")
nodes_per_group = gdf_unique_nodes.groupby('group')['node'].agg(['count', list]).head(10)
print(nodes_per_group)

,group,geometry,h3_index,node,Mode_Planned,Model,Line_Unique,address,area,location,...,Total_Unique_Lines,BRT Lines,Cable Line Lines,Funicular Lines,HighSpeed Rail Lines,Interurban Rail Lines,LRT Lines,Metro Lines,Suburban Rail Lines,centroid
0,0,"POLYGON ((35.583 33.21, 35.582 33.21, 35.581 3...",[8a2da4100a27fff],[31655],[Interurban Rail],[National],[['rail_1_1' 'rail_1_2']],"מנחם בגין/נורית, שדרות מנחם בגין, הורדים, קרית...",[צפון],[צפון],...,1,0,0,0,0,2,0,0,0,POINT (35.582 33.211)
1,1,"POLYGON ((35.327 32.926, 35.326 32.925, 35.325...",[8a2da4c22257fff],[36963],[BRT],[Haifa],[['brt022']],"מיי סנטר, מעלה כמון, אזור תעשייה כרמיאל, אזור ...",[צפון],[צפון],...,1,1,0,0,0,0,0,0,0,POINT (35.326 32.926)
2,2,"MULTIPOLYGON (((35.316 32.922, 35.316 32.922, ...","[8a2da4c22627fff, 8a2da4c2275ffff]","[36959, 36961]",[BRT],[Haifa],"[['brt021' 'brt022'], ['brt021']]","החרושת/חשמל, החרושת, אזור תעשייה כרמיאל, מרכז ...",[צפון],[צפון],...,2,3,0,0,0,0,0,0,0,POINT (35.317 32.923)
3,3,"POLYGON ((35.347 32.862, 35.346 32.862, 35.346...",[8a2da4c24987fff],[36924],[BRT],[Haifa],[['brt011' 'brt012' 'brt021' 'brt022']],"طريق الجليل, عرابة, נפת עכו, מחוז הצפון, 20176...",[צפון],[צפון],...,1,4,0,0,0,0,0,0,0,POINT (35.346 32.863)
4,4,"POLYGON ((35.282 32.911, 35.281 32.91, 35.28 3...",[8a2da4c30057fff],[36935],[BRT],[Haifa],[['brt021' 'brt022']],"בי""ס מקיף אורט כרמים, מעלה אורט, הגליל, רמת רב...",[צפון],[צפון],...,1,2,0,0,0,0,0,0,0,POINT (35.281 32.911)


Group demand totals:
   group  TotalDemand  TotalTransfers
0      0  3037.685341      153.393117
1      1   920.506646      920.506646
2      2   610.259136      119.767454
3      3   322.391024      322.391024
4      4   539.172726      453.679669

Total groups with demand data: 1013
Groups with non-zero TotalDemand: 858
Groups with non-zero TotalTransfers: 667

Unique nodes per group (first 10 groups):
       count                                        list
group                                                   
0          1                                     [31655]
1          1                                     [36963]
2          2                              [36959, 36961]
3          1                                     [36924]
4          1                                     [36935]
5          1                                     [36940]
6          1                                     [36938]
7          6  [36716, 36714, 36953, 36958, 36955, 36951]
8          1          

In [141]:
grouped_hubs[grouped_hubs['group']==583]['TotalDemand']

,TotalDemand
583,15047.0


In [142]:
gdf[gdf['group']==583]

,h3_index,node,Mode_Planned,Model,Line_Nunique,Line_Unique,geometry,group,address,area,location,TotalDemand,TotalTransfers
812,8a3f4dc10c37fff,523006,Metro,Tel Aviv,2,['Me03-L' 'Me03-R'],"POLYGON ((34.805 32.02, 34.805 32.02, 34.804 3...",583,"המלאכה, חולון, אזור תעשייה ב, חולון, נפת תל אב...",תל אביב,טבעת פנימית,13919.67,5851.43
813,8a3f4dc10d0ffff,525091,BRT,Tel Aviv,2,['BluRT1' 'BluRT2'],"POLYGON ((34.807 32.021, 34.807 32.021, 34.806...",583,"אחד במאי, שיכון גג, אזור, נפת תל אביב, מחוז תל...",תל אביב,טבעת פנימית,1127.33,1099.35


In [143]:
grouped_hubs.columns

Index(['group', 'geometry', 'h3_index', 'node', 'Mode_Planned', 'Model',
       'Line_Unique', 'address', 'area', 'location', 'TotalDemand',
       'TotalTransfers', 'Total_Unique_Lines', 'BRT Lines', 'Cable Line Lines',
       'Funicular Lines', 'HighSpeed Rail Lines', 'Interurban Rail Lines',
       'LRT Lines', 'Metro Lines', 'Suburban Rail Lines', 'centroid'],
      dtype='object')

In [144]:
grouped_hubs[grouped_hubs['group']==583][['TotalDemand','address','node']]

,TotalDemand,address,node
583,15047.0,"המלאכה, חולון, אזור תעשייה ב, חולון, נפת תל אב...","[523006, 525091]"


In [145]:
grouped_hubs.sort_values(by='LRT Lines', ascending=False)

,group,geometry,h3_index,node,Mode_Planned,Model,Line_Unique,address,area,location,...,Total_Unique_Lines,BRT Lines,Cable Line Lines,Funicular Lines,HighSpeed Rail Lines,Interurban Rail Lines,LRT Lines,Metro Lines,Suburban Rail Lines,centroid
292,292,"MULTIPOLYGON (((34.741 31.858, 34.741 31.858, ...","[8a3f4c3694cffff, 8a3f4c369617fff, 8a3f4c36962...","[512073, 511099, 511051, 511100, 511110, 51208...","[LRT, Interurban Rail, Suburban Rail]","[Ashdod-ashkelon, Tel Aviv, National]","[['Turquoise_N' 'Turquoise_S'], ['rail_5_1' 'r...","רמות בן צבי, יבנה, נפת רחובות, מחוז המרכז, ישראל",[תל אביב],[טבעת חיצונית],...,8,0,0,0,0,2,23,0,4,POINT (34.745 31.863)
393,393,"MULTIPOLYGON (((35.199 31.753, 35.199 31.754, ...","[8a3f4cb6ec17fff, 8a3f4cb6ec8ffff, 8a3f4cb6ecb...","[51048, 51157, 51136, 51142]",[LRT],[Jerusalem],[['LRT71' 'LRT72' 'LRT31' 'LRT32' 'LRT41' 'LRT...,"11, יהודה הנשיא, קטמונים (גוננים), ירושלים, נפ...",[ירושלים],[גלעין],...,3,0,0,0,0,0,22,0,0,POINT (35.2 31.755)
646,646,"MULTIPOLYGON (((34.774 32.059, 34.774 32.059, ...","[8a3f4dc1e52ffff, 8a3f4dc1e537fff, 8a3f4dc1ecd...","[511016, 513002, 513011]",[LRT],[Tel Aviv],[['Mgt1-E' 'Mgt1-W' 'Mgt2-E' 'Mgt2-W' 'Mgt3-E'...,"מקוה ישראל, לב תל-אביב, תל־אביב–יפו, נפת תל אב...",[תל אביב],[גלעין],...,2,0,0,0,0,0,18,0,0,POINT (34.774 32.062)
603,603,"MULTIPOLYGON (((34.776 32.037, 34.775 32.038, ...","[8a3f4dc13087fff, 8a3f4dc13087fff, 8a3f4dc1319...","[512055, 525099, 512026, 512026, 525097, 52509...","[LRT, BRT, Suburban Rail, Metro]","[Tel Aviv, National]","[['Me1-1N' 'Me1-1S' 'Me1-2N' 'Me1-2S'], ['BluR...","6, קמינסקה, קרית שלום, תל־אביב–יפו, נפת תל אבי...",[תל אביב],"[גלעין, טבעת פנימית]",...,5,5,0,0,0,0,18,4,4,POINT (34.777 32.038)
268,268,"MULTIPOLYGON (((34.652 31.811, 34.652 31.812, ...","[8a3f4c3590dffff, 8a3f4c3593b7fff, 8a3f4c3593b...","[511070, 511088, 512121, 511089, 512122]",[LRT],"[Tel Aviv, Ashdod-ashkelon]","[['RedN' 'RedS'], ['L1-N' 'L1-S' 'L2-N' 'L2-S']]","דוד בן גוריון, קריית איתנים, אשדוד, נפת אשקלון...",[תל אביב],[טבעת חיצונית],...,2,0,0,0,0,0,16,0,0,POINT (34.654 31.813)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
677,677,"POLYGON ((34.906 32.11, 34.905 32.109, 34.905 ...",[8a3f4dc2565ffff],[522017],[Metro],[Tel Aviv],[['Me02-W']],"חן הצפון, פתח תקווה, נפת פתח תקווה, מחוז המרכז...",[תל אביב],[טבעת תיכונה],...,1,0,0,0,0,0,0,1,0,POINT (34.905 32.11)
678,678,"POLYGON ((34.92 32.1, 34.919 32.1, 34.919 32.1...","[8a3f4dc2589ffff, 8a3f4dc25c6ffff]","[525512, 525582]",[BRT],[Tel Aviv],"[['RG-RH1' 'RG-RH2' 'RH-RG1' 'RH-RG2'], ['BRT-...","ראש העין, מועצה אזורית דרום השרון, נפת פתח תקו...",[תל אביב],[טבעת חיצונית],...,2,6,0,0,0,0,0,0,0,POINT (34.92 32.101)
679,679,"POLYGON ((34.864 32.145, 34.863 32.146, 34.863...","[8a3f4dc28417fff, 8a3f4dc28417fff]","[521068, 521068]","[BRT, Metro]",[Tel Aviv],"[['Me1-2N' 'Me1-2S'], ['Yelw-N' 'Yelw-S']]","תע""ש רמת השרון, תע""ש השרון, הוד השרון, נפת פתח...",[תל אביב],[טבעת תיכונה],...,2,2,0,0,0,0,0,2,0,POINT (34.864 32.146)
680,680,"POLYGON ((34.878 32.144, 34.877 32.145, 34.877...","[8a3f4dc28d27fff, 8a3f4dc28d27fff]","[521069, 521069]","[BRT, Metro]",[Tel Aviv],"[['Me1-2N' 'Me1-2S'], ['Yelw-N' 'Yelw-S']]","המחתרות, רמת הדר, הוד השרון, נפת פתח תקווה, מח...",[תל אביב],[טבעת תיכונה],...,2,2,0,0,0,0,0,2,0,POINT (34.878 32.145)


In [146]:
grouped_hubs.to_csv('/content/drive/MyDrive/Hubs/Grouped_Hubs_ReadyForPopEmp_29102025.csv', encoding='utf-8')

In [147]:
grouped_hubs[grouped_hubs['group']==583][['TotalDemand','TotalTransfers']]

,TotalDemand,TotalTransfers
583,15047.0,6950.78


In [148]:
node_demand_agg[node_demand_agg['node']=='400080']

NameError: name 'node_demand_agg' is not defined

# Glilot Darom - metro and rail far apart, need to adjust group id so they will be together.

In [149]:
grouped_hubs[grouped_hubs['group']==576]

,group,geometry,h3_index,node,Mode_Planned,Model,Line_Unique,address,area,location,...,Total_Unique_Lines,BRT Lines,Cable Line Lines,Funicular Lines,HighSpeed Rail Lines,Interurban Rail Lines,LRT Lines,Metro Lines,Suburban Rail Lines,centroid
576,576,"POLYGON ((34.809 32.134, 34.809 32.135, 34.809...","[8a3f4dc0e00ffff, 8a3f4dc0e00ffff, 8a3f4dc0e00...","[400018, 400018, 400018]","[HighSpeed Rail, Interurban Rail, Suburban Rail]",[National],[['rail_3_1' 'rail_3_2' 'rail_4_1' 'rail_4_2' ...,"איילון דרום, קרית שאול, רמת השרון, נפת תל אביב...",[תל אביב],[טבעת פנימית],...,3,0,0,0,6,8,0,0,14,POINT (34.809 32.135)


In [150]:
grouped_hubs.loc[grouped_hubs['group']==577, 'group'] = 576

In [152]:
gdf.loc[gdf['group']==577, 'group'] = 576

In [153]:
gdf[gdf['group']==576]

,h3_index,node,Mode_Planned,Model,Line_Nunique,Line_Unique,geometry,group,address,area,location,TotalDemand,TotalTransfers
802,8a3f4dc0e00ffff,400018,HighSpeed Rail,National,6,['rail_2_1' 'rail_2_2' 'rail_9_1' 'rail_9_2' '...,"POLYGON ((34.81 32.135, 34.809 32.134, 34.809 ...",576,"איילון דרום, קרית שאול, רמת השרון, נפת תל אביב...",תל אביב,טבעת פנימית,44964.57,44317.00
803,8a3f4dc0e00ffff,400018,Interurban Rail,National,8,['rail_3_1' 'rail_3_2' 'rail_4_1' 'rail_4_2' '...,"POLYGON ((34.81 32.135, 34.809 32.134, 34.809 ...",576,"איילון דרום, קרית שאול, רמת השרון, נפת תל אביב...",תל אביב,טבעת פנימית,44964.57,44317.00
804,8a3f4dc0e00ffff,400018,Suburban Rail,National,14,['rail_10_1' 'rail_10_2' 'rail_11_1' 'rail_11_...,"POLYGON ((34.81 32.135, 34.809 32.134, 34.809 ...",576,"איילון דרום, קרית שאול, רמת השרון, נפת תל אביב...",תל אביב,טבעת פנימית,44964.57,44317.00
805,8a3f4dc0e0a7fff,521063,Metro,Tel Aviv,4,['Me1-1N' 'Me1-1S' 'Me1-2N' 'Me1-2S'],"POLYGON ((34.809 32.131, 34.808 32.131, 34.808...",576,"איילון דרום, קרית שאול, תל־אביב–יפו, רמת השרון...",תל אביב,טבעת פנימית,116400.93,112902.25
806,8a3f4dc0e0a7fff,523019,Metro,Tel Aviv,2,['Me03-L' 'Me03-R'],"POLYGON ((34.809 32.131, 34.808 32.131, 34.808...",576,"איילון דרום, קרית שאול, תל־אביב–יפו, רמת השרון...",תל אביב,טבעת פנימית,105912.61,103290.13


Correct Code for grouping

In [154]:
# Ensure 'group' column is of integer type for consistent grouping
gdf['group'] = pd.to_numeric(gdf['group'], errors='coerce').fillna(-1).astype(int)

# Aggregate TotalDemand and TotalTransfers per unique node within each group *before* dissolving
node_demand_agg = gdf.groupby(['group', 'node']).agg({
    'TotalDemand': 'first',
    'TotalTransfers': 'first'
}).reset_index()

# Create a dictionary mapping group IDs to their total demand and transfers
group_demand_lookup = node_demand_agg.groupby('group').agg({
    'TotalDemand': 'sum',
    'TotalTransfers': 'sum'
}).to_dict('index')

# First, create aggregated columns in the original dataframe for the dissolve operation
def get_group_total_demand(group_id):
    return group_demand_lookup.get(group_id, {}).get('TotalDemand', 0)

def get_group_total_transfers(group_id):
    return group_demand_lookup.get(group_id, {}).get('TotalTransfers', 0)

# Add the aggregated totals as new columns
gdf['GroupTotalDemand'] = gdf['group'].apply(get_group_total_demand)
gdf['GroupTotalTransfers'] = gdf['group'].apply(get_group_total_transfers)

# Group the gdf by the 'group' column and dissolve geometries using dissolve()
grouped_hubs = gdf.dissolve(by='group', aggfunc={
    'h3_index': list,  # Keep all h3_index values as a list
    'node': lambda x: list(x), # Aggregate nodes into a list
    'Mode_Planned': lambda x: list(x.unique()), # Aggregate unique Mode_Planned into a list
    'Model': lambda x: list(x.unique()), # Aggregate unique Models into a list
    # Modified Line_Unique aggregation to return a flat list of unique lines
    'Line_Unique': lambda x: list(set([item for sublist in x for item in (sublist if isinstance(sublist, (list, np.ndarray)) else [sublist])])),
    'address': lambda x: x.iloc[0] if not x.isnull().all() else None, # Keep the first non-null address
    'area': lambda x: list(x.unique()), # Aggregate unique areas into a list
    'location': lambda x: list(x.unique()), # Aggregate unique locations into a list
    # Use the pre-calculated group totals (these will be the same for all rows in each group)
    'GroupTotalDemand': 'first',
    'GroupTotalTransfers': 'first'
}).reset_index() # Reset index to make 'group' a column

# Rename the columns to the desired names
grouped_hubs = grouped_hubs.rename(columns={
    'GroupTotalDemand': 'TotalDemand',
    'GroupTotalTransfers': 'TotalTransfers'
})

# Convert the result to a GeoDataFrame (dissolve already returns a GeoDataFrame, but good practice to be explicit)
grouped_hubs = gpd.GeoDataFrame(grouped_hubs, geometry='geometry', crs=gdf.crs)

# Calculate the number of unique lines per group based on the aggregated 'Line_Unique' list
grouped_hubs['Total_Unique_Lines'] = grouped_hubs['Line_Unique'].apply(lambda x: len(x) if isinstance(x, list) else 0)

# Calculate sum of Line_Nunique per Mode_Planned per group
# This part remains useful for seeing the breakdown by mode within the group
line_nunique_by_mode = gdf.groupby(['group', 'Mode_Planned'])['Line_Nunique'].sum().unstack(fill_value=0)

# Rename columns to desired format (e.g., 'BRT_Line_Nunique')
line_nunique_by_mode.columns = [f'{col} Lines' for col in line_nunique_by_mode.columns]

# Merge the new Line_Nunique columns back into grouped_hubs
grouped_hubs = grouped_hubs.merge(line_nunique_by_mode, left_on='group', right_index=True, how='left')

# Create centroid for each group's geometry
# Ensure the GeoDataFrame is in a projected CRS before calculating centroids for accurate results
if grouped_hubs.crs.is_geographic:
    grouped_hubs = grouped_hubs.to_crs(epsg=crs_il) # Reproject to a projected CRS for Israel

grouped_hubs['centroid'] = grouped_hubs.geometry.centroid

# Clean up the temporary columns from the original dataframe
gdf = gdf.drop(columns=['GroupTotalDemand', 'GroupTotalTransfers'])

# Display the head of the new grouped_hubs GeoDataFrame
display(grouped_hubs.head())

,group,geometry,h3_index,node,Mode_Planned,Model,Line_Unique,address,area,location,...,Total_Unique_Lines,BRT Lines,Cable Line Lines,Funicular Lines,HighSpeed Rail Lines,Interurban Rail Lines,LRT Lines,Metro Lines,Suburban Rail Lines,centroid
0,0,"POLYGON ((35.583 33.21, 35.582 33.21, 35.581 3...",[8a2da4100a27fff],[31655],[Interurban Rail],[National],[['rail_1_1' 'rail_1_2']],"מנחם בגין/נורית, שדרות מנחם בגין, הורדים, קרית...",[צפון],[צפון],...,1,0,0,0,0,2,0,0,0,POINT (35.582 33.211)
1,1,"POLYGON ((35.327 32.926, 35.326 32.925, 35.325...",[8a2da4c22257fff],[36963],[BRT],[Haifa],[['brt022']],"מיי סנטר, מעלה כמון, אזור תעשייה כרמיאל, אזור ...",[צפון],[צפון],...,1,1,0,0,0,0,0,0,0,POINT (35.326 32.926)
2,2,"MULTIPOLYGON (((35.316 32.922, 35.316 32.922, ...","[8a2da4c22627fff, 8a2da4c2275ffff]","[36959, 36961]",[BRT],[Haifa],"[['brt021' 'brt022'], ['brt021']]","החרושת/חשמל, החרושת, אזור תעשייה כרמיאל, מרכז ...",[צפון],[צפון],...,2,3,0,0,0,0,0,0,0,POINT (35.317 32.923)
3,3,"POLYGON ((35.347 32.862, 35.346 32.862, 35.346...",[8a2da4c24987fff],[36924],[BRT],[Haifa],[['brt011' 'brt012' 'brt021' 'brt022']],"طريق الجليل, عرابة, נפת עכו, מחוז הצפון, 20176...",[צפון],[צפון],...,1,4,0,0,0,0,0,0,0,POINT (35.346 32.863)
4,4,"POLYGON ((35.282 32.911, 35.281 32.91, 35.28 3...",[8a2da4c30057fff],[36935],[BRT],[Haifa],[['brt021' 'brt022']],"בי""ס מקיף אורט כרמים, מעלה אורט, הגליל, רמת רב...",[צפון],[צפון],...,1,2,0,0,0,0,0,0,0,POINT (35.281 32.911)


In [155]:
grouped_hubs[grouped_hubs['group']==576]

,group,geometry,h3_index,node,Mode_Planned,Model,Line_Unique,address,area,location,...,Total_Unique_Lines,BRT Lines,Cable Line Lines,Funicular Lines,HighSpeed Rail Lines,Interurban Rail Lines,LRT Lines,Metro Lines,Suburban Rail Lines,centroid
576,576,"MULTIPOLYGON (((34.808 32.131, 34.808 32.132, ...","[8a3f4dc0e00ffff, 8a3f4dc0e00ffff, 8a3f4dc0e00...","[400018, 400018, 400018, 521063, 523019]","[HighSpeed Rail, Interurban Rail, Suburban Rai...","[National, Tel Aviv]","[['Me03-L' 'Me03-R'], ['Me1-1N' 'Me1-1S' 'Me1-...","איילון דרום, קרית שאול, רמת השרון, נפת תל אביב...",[תל אביב],[טבעת פנימית],...,5,0,0,0,6,8,0,6,14,POINT (34.809 32.133)


# correct: HighSpeed Rail in Savidor!

In [156]:
grouped_hubs.columns

Index(['group', 'geometry', 'h3_index', 'node', 'Mode_Planned', 'Model',
       'Line_Unique', 'address', 'area', 'location', 'TotalDemand',
       'TotalTransfers', 'Total_Unique_Lines', 'BRT Lines', 'Cable Line Lines',
       'Funicular Lines', 'HighSpeed Rail Lines', 'Interurban Rail Lines',
       'LRT Lines', 'Metro Lines', 'Suburban Rail Lines', 'centroid'],
      dtype='object')

In [157]:
grouped_hubs.to_csv('/content/drive/MyDrive/Hubs/Grouped_Hubs_ReadyForPopEmp_29102025.csv', encoding='utf-8')

In [158]:
grouped_hubs = pd.read_csv('/content/drive/MyDrive/Hubs/Grouped_Hubs_ReadyForPopEmp_29102025.csv', encoding='utf-8')

In [159]:
grouped_hubs[grouped_hubs['group']==955]

,Unnamed: 0,group,geometry,h3_index,node,Mode_Planned,Model,Line_Unique,address,area,...,Total_Unique_Lines,BRT Lines,Cable Line Lines,Funicular Lines,HighSpeed Rail Lines,Interurban Rail Lines,LRT Lines,Metro Lines,Suburban Rail Lines,centroid
954,954,955,MULTIPOLYGON (((34.77596964028979 31.887242985...,"['8a3f4dd9a14ffff', '8a3f4dd9a14ffff', '8a3f4d...","['511075', '511075', '512110', '511068', '5122...",['LRT'],"['Tel Aviv', 'Ashdod-ashkelon']","[""['L1-N' 'L1-S']"", ""['Light-Blue_S' 'Turquois...","מועצה אזורית גן רווה, נפת רחובות, מחוז המרכז, ...",['תל אביב'],...,2,0,0,0,0,0,12,0,0,POINT (34.778608391982594 31.888777676526423)


In [160]:
grouped_hubs[grouped_hubs['group']==955]['Line_Unique'].values

array(['["[\'L1-N\' \'L1-S\']", "[\'Light-Blue_S\' \'Turquoise_N\' \'Turquoise_S\']"]'],
      dtype=object)

In [161]:
grouped_hubs[grouped_hubs['group']==955]['Mode_Planned'].values

array(["['LRT']"], dtype=object)

In [162]:
grouped_hubs[grouped_hubs['group']==921]['Line_Unique'].values

array(['["[\'Grn2-N\']", "[\'BluRT1\' \'BluRT2\']"]'], dtype=object)

In [163]:
grouped_hubs[grouped_hubs['group']==920]['Line_Unique'].values

array(['["[\'BluRT1\' \'BluRT2\']"]'], dtype=object)

In [14]:
# Load the CSV file
import folium
from shapely import wkt
import geopandas as gpd
import pandas as pd

all_nodes_file = '/content/drive/MyDrive/Hubs/All_nodes+lines_21082025.csv'
encoding = 'windows-1255'
crs_il = 2039

all_nodes_df = pd.read_csv(all_nodes_file, encoding=encoding)

# Convert to GeoDataFrame, assuming 'geometry' column is in WKT format
# Use errors='coerce' to handle invalid WKT strings
all_nodes_df['geometry'] = all_nodes_df['geometry'].apply(lambda geom: wkt.loads(geom) if isinstance(geom, str) else None)
all_nodes_gdf = gpd.GeoDataFrame(all_nodes_df, geometry='geometry', crs=crs_il)

# Reproject to WGS84 (EPSG:4326) for Folium
all_nodes_gdf_wgs84 = all_nodes_gdf.to_crs(epsg=4326)

# Create a Folium map centered around the data
# Filter out rows with None geometry before calculating the centroid of the union
valid_geometries = all_nodes_gdf_wgs84.dropna(subset=['geometry']).geometry
map_center = valid_geometries.unary_union.centroid.coords[0][::-1]
m = folium.Map(location=map_center, zoom_start=8)

# Add markers to the map with tooltips
for index, row in all_nodes_gdf_wgs84.iterrows():
    if row.geometry is not None: # Check if geometry is not None after loading
        folium.Marker(
            location=[row.geometry.y, row.geometry.x],
            tooltip=f"LINE_ID: {row['LINE_ID']}<br>Node: {row['node']}"
        ).add_to(m)

# Display the map
m

Output hidden; open in https://colab.research.google.com to view.